<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_4/All-Examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# Phi-3 Mini 4K Instruct — Оценка на датасете (качество + скорость)
# ================================================================

!pip install evaluate sacrebleu rouge-score -q

# ----------------------------------------------------------------
# 1. Импорты
# ----------------------------------------------------------------
import os
import gc
import re
import time
import torch
import warnings
from typing import List, Dict, Any

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForCausalLM,
    pipeline,
    GenerationConfig,
)

# Для метрик качества (попробуем импортировать, если есть)
try:
    import evaluate
    from evaluate import load
    HAS_EVALUATE = True
except ImportError:
    HAS_EVALUATE = False
    print("⚠️ Библиотека 'evaluate' не установлена. Метрики BLEU/ROUGE будут недоступны.")
    print("   Установите: pip install evaluate sacrebleu rouge-score")

try:
    from rouge_score import rouge_scorer
    HAS_ROUGE = True
except ImportError:
    HAS_ROUGE = False
    print("⚠️ 'rouge-score' не установлена. ROUGE будет пропущен.")
    print("   Установите: pip install rouge-score")

# Подавляем предупреждение о GenerationMixin
warnings.filterwarnings("ignore", message=".*Phi3ForCausalLM has generative capabilities.*")

# ----------------------------------------------------------------
# 2. Основные настройки
# ----------------------------------------------------------------
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.2
TOP_P = 0.9
SEED = 42

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------------------------------------------------------
# 3. Проверка окружения
# ----------------------------------------------------------------
print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)

print(f"PyTorch:       {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU:           {gpu_name}")
    print(f"VRAM:          {gpu_memory:.2f} GB")
    DTYPE = torch.float16
    print(f"Dtype:         {DTYPE}")
    torch.cuda.reset_peak_memory_stats()
else:
    print("⚠️ GPU не обнаружен. Модель будет загружена на CPU.")
    DTYPE = torch.float32

print("=" * 70)

# ----------------------------------------------------------------
# 4. Освобождение памяти
# ----------------------------------------------------------------
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----------------------------------------------------------------
# 5. Загрузка токенизатора и модели
# ----------------------------------------------------------------
print("\n🔹 Загружаем токенизатор...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("✅ Токенизатор загружен")
print(f"Vocabulary size: {len(tokenizer):,}")
print(f"EOS token:       {tokenizer.eos_token!r}")
print(f"PAD token:       {tokenizer.pad_token!r}")

print("\n🔹 Загружаем конфигурацию модели...")
config = AutoConfig.from_pretrained(MODEL_NAME)
print("✅ Конфигурация загружена")

print("\n🔹 Загружаем модель:")
print(f"   {MODEL_NAME}")

model_kwargs = {
    "config": config,
    "torch_dtype": DTYPE,
}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"
else:
    model_kwargs["device_map"] = "cpu"

start_load_time = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
load_time = time.time() - start_load_time

model.eval()
print(f"✅ Модель загружена за {load_time:.2f} сек.")

# ----------------------------------------------------------------
# 6. Информация о модели
# ----------------------------------------------------------------
num_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "=" * 70)
print("ИНФОРМАЦИЯ О МОДЕЛИ")
print("=" * 70)
print(f"Всего параметров:      {num_parameters / 1e9:.3f} B")
print(f"Обучаемых параметров:  {trainable_parameters / 1e9:.3f} B")
print(f"Device map: {getattr(model, 'hf_device_map', 'N/A')}")
if torch.cuda.is_available():
    current_vram = torch.cuda.memory_allocated() / 1024**3
    peak_vram = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Занято VRAM:          {current_vram:.2f} GB")
    print(f"Пиковое VRAM:         {peak_vram:.2f} GB")
print("=" * 70)

# ----------------------------------------------------------------
# 7. Pipeline
# ----------------------------------------------------------------
generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
)
print("\n✅ Pipeline успешно создан")

# ----------------------------------------------------------------
# 8. Функция генерации
# ----------------------------------------------------------------
def generate_text(
    prompt: str,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
    do_sample: bool = True,
) -> Dict[str, Any]:
    """Генерирует ответ и возвращает текст + метрики производительности."""
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt должен быть непустой строкой")

    messages = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, "apply_chat_template"):
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        formatted_prompt = prompt

    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else 1.0,
        top_p=top_p if do_sample else 1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    start_time = time.time()
    with torch.inference_mode():
        result = generator(
            formatted_prompt,
            return_full_text=False,
            generation_config=gen_config,
            clean_up_tokenization_spaces=False,
        )
    end_time = time.time()

    generated_text = result[0]["generated_text"].strip() if result else ""
    generation_time = end_time - start_time

    input_tokens = len(tokenizer.encode(formatted_prompt))
    output_tokens = len(tokenizer.encode(generated_text))
    tokens_per_second = output_tokens / generation_time if generation_time > 0 else 0.0

    return {
        "text": generated_text,
        "metrics": {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "generation_time_sec": round(generation_time, 3),
            "tokens_per_second": round(tokens_per_second, 2),
        }
    }

# ----------------------------------------------------------------
# 9. УЛУЧШЕННАЯ ФУНКЦИЯ ИЗВЛЕЧЕНИЯ ЧИСЛА
# ----------------------------------------------------------------
def extract_number(text: str) -> str:
    """
    Извлекает число из текста, отдавая предпочтение числу после знака '=' или 'равно'.
    Если таких нет, возвращает последнее число в тексте.
    """
    text = text.replace(",", "")
    # Ищем число после '='
    if "=" in text:
        parts = text.split("=")
        if len(parts) > 1:
            after_eq = parts[-1]
            nums = re.findall(r"[-+]?\d*\.?\d+", after_eq)
            if nums:
                return nums[0]
    # Ищем после 'равно'
    if "равно" in text:
        parts = text.split("равно")
        if len(parts) > 1:
            after_eq = parts[-1]
            nums = re.findall(r"[-+]?\d*\.?\d+", after_eq)
            if nums:
                return nums[0]
    # Иначе все числа
    numbers = re.findall(r"[-+]?\d*\.?\d+", text)
    return numbers[-1] if numbers else ""

# ----------------------------------------------------------------
# 10. ДАТАСЕТ (расширенный)
# ----------------------------------------------------------------
dataset = [
    # Переводы (русский → английский)
    {"type": "translation", "input": "Переведи на английский: Привет, как дела?", "target": "Hello, how are you?"},
    {"type": "translation", "input": "Переведи на английский: Сегодня отличная погода.", "target": "Today is great weather."},
    {"type": "translation", "input": "Переведи на английский: Я люблю программирование.", "target": "I love programming."},
    {"type": "translation", "input": "Переведи на английский: Который час?", "target": "What time is it?"},
    {"type": "translation", "input": "Переведи на английский: Это очень интересно.", "target": "This is very interesting."},
    # Математика
    {"type": "math", "input": "Сколько будет 2 + 2? Ответ дай числом.", "target": "4"},
    {"type": "math", "input": "Сколько будет 5 * 3? Ответ дай числом.", "target": "15"},
    {"type": "math", "input": "Сколько будет 10 - 4? Ответ дай числом.", "target": "6"},
    {"type": "math", "input": "Сколько будет 12 / 3? Ответ дай числом.", "target": "4"},
    {"type": "math", "input": "Сколько будет 7 + 8? Ответ дай числом.", "target": "15"},
]

print("\n" + "=" * 70)
print("ДАТАСЕТ ДЛЯ ОЦЕНКИ")
print("=" * 70)
for i, ex in enumerate(dataset, 1):
    print(f"{i}. [{ex['type']}] {ex['input']} -> {ex['target']}")
print("=" * 70)

# ----------------------------------------------------------------
# 11. ЗАГРУЗКА МЕТРИК (если доступны)
# ----------------------------------------------------------------
bleu_metric = None
rouge_scorer_obj = None

if HAS_EVALUATE:
    try:
        bleu_metric = load("bleu")
    except Exception as e:
        print(f"⚠️ Не удалось загрузить BLEU: {e}")

if HAS_ROUGE:
    rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# ----------------------------------------------------------------
# 12. ФУНКЦИЯ ОЦЕНКИ
# ----------------------------------------------------------------
def evaluate_dataset(dataset: List[Dict]) -> Dict:
    """
    Прогоняет все примеры, собирает предсказания,
    вычисляет метрики качества и производительности.
    """
    predictions = []
    references = []
    example_types = []
    perf_metrics = []

    # Для накопления метрик качества
    bleu_scores = []
    rouge_scores = []
    math_correct = 0
    math_total = 0

    print("\n" + "=" * 70)
    print("ЗАПУСК ОЦЕНКИ НА ДАТАСЕТЕ")
    print("=" * 70)

    for idx, example in enumerate(dataset, 1):
        print(f"\n--- Пример {idx} ({example['type']}) ---")
        print(f"Вопрос: {example['input']}")
        print(f"Эталон: {example['target']}")

        # Генерация
        result = generate_text(example["input"], max_new_tokens=MAX_NEW_TOKENS)
        pred_text = result["text"]
        metrics = result["metrics"]

        print(f"Ответ модели: {pred_text}")

        # Сохраняем для метрик
        predictions.append(pred_text)
        references.append([example["target"]])   # для BLEU
        example_types.append(example["type"])
        perf_metrics.append(metrics)

        # Вывод скорости
        print(f"   ⏱ Время: {metrics['generation_time_sec']} сек, "
              f"токенов: {metrics['output_tokens']}, TPS: {metrics['tokens_per_second']}")

        # ----- Сбор метрик качества по типам -----
        if example["type"] == "translation":
            # BLEU (накапливаем для среднего)
            if bleu_metric is not None:
                try:
                    score = bleu_metric.compute(predictions=[pred_text], references=[[example["target"]]])
                    bleu_scores.append(score["bleu"])
                except Exception as e:
                    print(f"   BLEU ошибка: {e}")

            # ROUGE-L
            if rouge_scorer_obj is not None:
                try:
                    scores = rouge_scorer_obj.score(example["target"], pred_text)
                    rouge_scores.append(scores['rougeL'].fmeasure)
                except Exception as e:
                    print(f"   ROUGE ошибка: {e}")

        elif example["type"] == "math":
            math_total += 1
            pred_num = extract_number(pred_text)
            ref_num = extract_number(example["target"])
            if pred_num and ref_num and pred_num == ref_num:
                math_correct += 1
                print("   ✅ Математика: верно")
            else:
                print(f"   ❌ Математика: неверно (извлечено '{pred_num}', ожидалось '{ref_num}')")

    # ----- ИТОГОВЫЕ МЕТРИКИ КАЧЕСТВА -----
    print("\n" + "=" * 70)
    print("РЕЗУЛЬТАТЫ ОЦЕНКИ КАЧЕСТВА")
    print("=" * 70)

    # Средний BLEU по переводам
    if bleu_scores:
        avg_bleu = sum(bleu_scores) / len(bleu_scores)
        print(f"📊 Средний BLEU (переводы): {avg_bleu:.3f}")
    else:
        print("📊 BLEU: не доступен (установите evaluate и sacrebleu)")

    # Средний ROUGE-L
    if rouge_scores:
        avg_rouge = sum(rouge_scores) / len(rouge_scores)
        print(f"📊 Средний ROUGE-L (переводы): {avg_rouge:.3f}")
    else:
        print("📊 ROUGE-L: не доступен (установите rouge-score)")

    # Точность на математике
    if math_total > 0:
        accuracy = math_correct / math_total
        print(f"📊 Точность на математике: {accuracy:.2%} ({math_correct}/{math_total})")
    else:
        print("📊 Математических задач нет.")

    # ----- СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ -----
    print("\n" + "=" * 70)
    print("СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ")
    print("=" * 70)
    if perf_metrics:
        avg_input = sum(m["input_tokens"] for m in perf_metrics) / len(perf_metrics)
        avg_output = sum(m["output_tokens"] for m in perf_metrics) / len(perf_metrics)
        avg_time = sum(m["generation_time_sec"] for m in perf_metrics) / len(perf_metrics)
        avg_tps = sum(m["tokens_per_second"] for m in perf_metrics) / len(perf_metrics)
        print(f"Среднее токенов в запросе:      {avg_input:.1f}")
        print(f"Среднее сгенерировано токенов:  {avg_output:.1f}")
        print(f"Среднее время генерации:        {avg_time:.3f} сек.")
        print(f"Средняя скорость (TPS):         {avg_tps:.2f} токен/сек.")

    return {
        "predictions": predictions,
        "references": references,
        "perf_metrics": perf_metrics,
    }

# ----------------------------------------------------------------
# 13. ЗАПУСК
# ----------------------------------------------------------------
if __name__ == "__main__":
    results = evaluate_dataset(dataset)
    print("\n" + "=" * 70)
    print("✅ ОЦЕНКА ЗАВЕРШЕНА")
    print("=" * 70)

In [ ]:
# ================================================================
# Phi-3 Mini 4K Instruct — Few-shot (2 примера) с системной инструкцией
# ================================================================

!pip install evaluate sacrebleu rouge-score -q

import os, gc, re, time, torch, warnings
from typing import List, Dict, Any, Tuple
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForCausalLM,
    pipeline, GenerationConfig
)

# Подавляем предупреждения о clean_up_tokenization_spaces
warnings.filterwarnings("ignore", message=".*clean_up_tokenization_spaces.*")
warnings.filterwarnings("ignore", message=".*Phi3ForCausalLM has generative capabilities.*")

# Импорт метрик
try:
    import evaluate
    from evaluate import load
    HAS_EVALUATE = True
except ImportError:
    HAS_EVALUATE = False
    print("⚠️ Установите: pip install evaluate sacrebleu")

try:
    from rouge_score import rouge_scorer
    HAS_ROUGE = True
except ImportError:
    HAS_ROUGE = False
    print("⚠️ Установите: pip install rouge-score")

# ----------------------------------------------------------------
# Настройки
# ----------------------------------------------------------------
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
MAX_NEW_TOKENS = 64            # достаточно для коротких ответов
TEMPERATURE = 0.2
TOP_P = 0.9
SEED = 42
NUM_DEMOS = 2                  # количество демонстраций для каждого типа

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------------------------------------------------------
# Окружение
# ----------------------------------------------------------------
print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    DTYPE = torch.float16
    torch.cuda.reset_peak_memory_stats()
else:
    DTYPE = torch.float32
print("=" * 70)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----------------------------------------------------------------
# Загрузка модели
# ----------------------------------------------------------------
print("\n🔹 Загружаем токенизатор...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("✅ Токенизатор загружен")

print("\n🔹 Загружаем модель...")
model_kwargs = {
    "config": AutoConfig.from_pretrained(MODEL_NAME),
    "torch_dtype": DTYPE,
}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"

start = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
load_time = time.time() - start
model.eval()
print(f"✅ Модель загружена за {load_time:.2f} сек.")

# Информация о модели
print("\n" + "=" * 70)
print("ИНФОРМАЦИЯ О МОДЕЛИ")
print("=" * 70)
print(f"Всего параметров: {sum(p.numel() for p in model.parameters()) / 1e9:.3f} B")
if torch.cuda.is_available():
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("=" * 70)

# Pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("\n✅ Pipeline создан")

# ----------------------------------------------------------------
# Вспомогательные функции
# ----------------------------------------------------------------
def extract_number(text: str) -> str:
    """Извлекает последнее число или число после '=' / 'равно'."""
    text = text.replace(",", "")
    if "=" in text:
        parts = text.split("=")
        if len(parts) > 1:
            nums = re.findall(r"[-+]?\d*\.?\d+", parts[-1])
            if nums:
                return nums[0]
    if "равно" in text:
        parts = text.split("равно")
        if len(parts) > 1:
            nums = re.findall(r"[-+]?\d*\.?\d+", parts[-1])
            if nums:
                return nums[0]
    nums = re.findall(r"[-+]?\d*\.?\d+", text)
    return nums[-1] if nums else ""

def clean_response(text: str) -> str:
    """Обрезает ответ при появлении нового вопроса или маркера."""
    markers = ["Вопрос:", "Сколько будет", "Переведи", "?"]
    for marker in markers:
        idx = text.find(marker)
        if idx != -1 and idx > 0:  # если маркер не в самом начале
            return text[:idx].strip()
    return text.strip()

# ----------------------------------------------------------------
# Демонстрации (по NUM_DEMOS примеров на тип)
# ----------------------------------------------------------------
DEMO_TRANSLATIONS: List[Tuple[str, str]] = [
    ("Переведи на английский: Привет, как дела?", "Hello, how are you?"),
    ("Переведи на английский: Сегодня отличная погода.", "Today is great weather."),
]

DEMO_MATH: List[Tuple[str, str]] = [
    ("Сколько будет 2 + 2? Ответ дай числом.", "4"),
    ("Сколько будет 5 * 3? Ответ дай числом.", "15"),
]

# ----------------------------------------------------------------
# Тестовый датасет (без демонстраций)
# ----------------------------------------------------------------
test_dataset = [
    {"type": "translation", "input": "Переведи на английский: Я люблю программирование.", "target": "I love programming."},
    {"type": "translation", "input": "Переведи на английский: Который час?", "target": "What time is it?"},
    {"type": "translation", "input": "Переведи на английский: Это очень интересно.", "target": "This is very interesting."},
    {"type": "math", "input": "Сколько будет 10 - 4? Ответ дай числом.", "target": "6"},
    {"type": "math", "input": "Сколько будет 12 / 3? Ответ дай числом.", "target": "4"},
    {"type": "math", "input": "Сколько будет 7 + 8? Ответ дай числом.", "target": "15"},
]

print("\n" + "=" * 70)
print("ТЕСТОВЫЙ ДАТАСЕТ")
print("=" * 70)
for i, ex in enumerate(test_dataset, 1):
    print(f"{i}. [{ex['type']}] {ex['input']} -> {ex['target']}")
print("=" * 70)

# ----------------------------------------------------------------
# Генерация с Few-shot (2 примера) + системное сообщение
# ----------------------------------------------------------------
def generate_fewshot(
    question: str,
    demonstrations: List[Tuple[str, str]],
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> Dict[str, Any]:
    """
    Формирует диалог:
    - system: краткая инструкция (не задавай вопросов, отвечай кратко)
    - user/assistant пары демонстраций
    - user: текущий вопрос
    """
    # Системное сообщение (поддерживается в Phi-3)
    system_msg = "Ты — полезный ассистент. Отвечай кратко и только на последний вопрос. Не задавай новых вопросов."

    messages = [{"role": "system", "content": system_msg}]

    for inp, out in demonstrations:
        messages.append({"role": "user", "content": inp})
        messages.append({"role": "assistant", "content": out})

    messages.append({"role": "user", "content": question})

    # Применяем чат-шаблон
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    start_time = time.time()
    with torch.inference_mode():
        result = generator(formatted, return_full_text=False, generation_config=gen_config)
    end_time = time.time()

    raw = result[0]["generated_text"].strip()
    # Пост-обработка
    cleaned = clean_response(raw)
    if not cleaned:
        cleaned = raw

    input_tokens = len(tokenizer.encode(formatted))
    output_tokens = len(tokenizer.encode(cleaned))
    gen_time = end_time - start_time
    tps = output_tokens / gen_time if gen_time > 0 else 0.0

    return {
        "text": cleaned,
        "metrics": {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "generation_time_sec": round(gen_time, 3),
            "tokens_per_second": round(tps, 2),
        }
    }

# ----------------------------------------------------------------
# Загрузка метрик
# ----------------------------------------------------------------
bleu_metric = load("bleu") if HAS_EVALUATE else None
rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True) if HAS_ROUGE else None

# ----------------------------------------------------------------
# Оценка
# ----------------------------------------------------------------
def evaluate_fewshot(test_dataset):
    predictions, refs, types, perf_metrics = [], [], [], []
    bleu_scores, rouge_scores = [], []
    math_correct, math_total = 0, 0

    print("\n" + "=" * 70)
    print("ЗАПУСК ОЦЕНКИ (FEW-SHOT, 2 демонстрации)")
    print("=" * 70)

    for idx, ex in enumerate(test_dataset, 1):
        ex_type = ex["type"]
        print(f"\n--- Пример {idx} ({ex_type}) ---")
        print(f"Вопрос: {ex['input']}")
        print(f"Эталон: {ex['target']}")

        demos = DEMO_TRANSLATIONS if ex_type == "translation" else DEMO_MATH
        result = generate_fewshot(ex["input"], demos)
        pred = result["text"]
        metrics = result["metrics"]

        print(f"Ответ модели: {pred}")
        print(f"   ⏱ {metrics['generation_time_sec']} сек, {metrics['output_tokens']} токенов, TPS: {metrics['tokens_per_second']}")

        predictions.append(pred)
        refs.append([ex["target"]])
        types.append(ex_type)
        perf_metrics.append(metrics)

        # Качество
        if ex_type == "translation":
            if bleu_metric:
                try:
                    score = bleu_metric.compute(predictions=[pred], references=[[ex["target"]]])
                    bleu_scores.append(score["bleu"])
                except Exception as e:
                    pass
            if rouge_scorer_obj:
                try:
                    scores = rouge_scorer_obj.score(ex["target"], pred)
                    rouge_scores.append(scores['rougeL'].fmeasure)
                except:
                    pass
        elif ex_type == "math":
            math_total += 1
            pred_num = extract_number(pred)
            ref_num = extract_number(ex["target"])
            if pred_num and ref_num and pred_num == ref_num:
                math_correct += 1
                print("   ✅ Математика верно")
            else:
                print(f"   ❌ Математика: извлечено '{pred_num}', ожидалось '{ref_num}'")

    # Итоги качества
    print("\n" + "=" * 70)
    print("РЕЗУЛЬТАТЫ КАЧЕСТВА")
    print("=" * 70)
    if bleu_scores:
        print(f"📊 Средний BLEU: {sum(bleu_scores)/len(bleu_scores):.3f}")
    else:
        print("📊 BLEU: не доступен (установите sacrebleu)")

    if rouge_scores:
        print(f"📊 Средний ROUGE-L: {sum(rouge_scores)/len(rouge_scores):.3f}")

    if math_total:
        print(f"📊 Точность на математике: {math_correct/math_total:.2%} ({math_correct}/{math_total})")

    # Средние метрики производительности
    print("\n" + "=" * 70)
    print("СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ")
    print("=" * 70)
    if perf_metrics:
        avg_in = sum(m["input_tokens"] for m in perf_metrics) / len(perf_metrics)
        avg_out = sum(m["output_tokens"] for m in perf_metrics) / len(perf_metrics)
        avg_time = sum(m["generation_time_sec"] for m in perf_metrics) / len(perf_metrics)
        avg_tps = sum(m["tokens_per_second"] for m in perf_metrics) / len(perf_metrics)
        print(f"Среднее токенов в запросе:   {avg_in:.1f}")
        print(f"Среднее сгенерировано токенов: {avg_out:.1f}")
        print(f"Среднее время генерации:     {avg_time:.3f} сек.")
        print(f"Средняя скорость (TPS):      {avg_tps:.2f} токен/сек.")

    return {"predictions": predictions, "perf_metrics": perf_metrics}

# ----------------------------------------------------------------
# Запуск
# ----------------------------------------------------------------
if __name__ == "__main__":
    results = evaluate_fewshot(test_dataset)
    print("\n" + "=" * 70)
    print("✅ ОЦЕНКА FEW-SHOT ЗАВЕРШЕНА")
    print("=" * 70)

In [ ]:
# ================================================================
# Phi-3 Mini 4K Instruct — Сравнение режимов: Few-shot, CoT, Self-Consistency
# Сложные математические демонстрации для CoT
# ================================================================

!pip install evaluate sacrebleu rouge-score -q

import os, gc, re, time, torch, warnings
from typing import List, Dict, Any, Tuple
from collections import Counter
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForCausalLM,
    pipeline, GenerationConfig
)

warnings.filterwarnings("ignore", message=".*clean_up_tokenization_spaces.*")
warnings.filterwarnings("ignore", message=".*Phi3ForCausalLM has generative capabilities.*")

try:
    import evaluate
    from evaluate import load
    HAS_EVALUATE = True
except ImportError:
    HAS_EVALUATE = False

try:
    from rouge_score import rouge_scorer
    HAS_ROUGE = True
except ImportError:
    HAS_ROUGE = False

# ----------------------------------------------------------------
# Настройки
# ----------------------------------------------------------------
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
MAX_NEW_TOKENS = 512            # для CoT нужно больше токенов
TEMPERATURE = 0.7
TOP_P = 0.9
SEED = 42
NUM_CONSISTENCY = 5             # количество генераций для Self-Consistency
USE_SYSTEM = True

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------------------------------------------------------
# Окружение
# ----------------------------------------------------------------
print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    DTYPE = torch.float16
    torch.cuda.reset_peak_memory_stats()
else:
    DTYPE = torch.float32
print("=" * 70)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----------------------------------------------------------------
# Загрузка модели
# ----------------------------------------------------------------
print("\n🔹 Загружаем токенизатор...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("✅ Токенизатор загружен")

print("\n🔹 Загружаем модель...")
model_kwargs = {
    "config": AutoConfig.from_pretrained(MODEL_NAME),
    "torch_dtype": DTYPE,
}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"

start = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
load_time = time.time() - start
model.eval()
print(f"✅ Модель загружена за {load_time:.2f} сек.")

print("\n" + "=" * 70)
print("ИНФОРМАЦИЯ О МОДЕЛИ")
print("=" * 70)
print(f"Всего параметров: {sum(p.numel() for p in model.parameters()) / 1e9:.3f} B")
if torch.cuda.is_available():
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("=" * 70)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("\n✅ Pipeline создан")

# ----------------------------------------------------------------
# Вспомогательные функции
# ----------------------------------------------------------------
def extract_number(text: str) -> str:
    """Извлекает последнее число или число после '=' / 'равно'."""
    text = text.replace(",", "")
    if "=" in text:
        parts = text.split("=")
        if len(parts) > 1:
            nums = re.findall(r"[-+]?\d*\.?\d+", parts[-1])
            if nums:
                return nums[0]
    if "равно" in text:
        parts = text.split("равно")
        if len(parts) > 1:
            nums = re.findall(r"[-+]?\d*\.?\d+", parts[-1])
            if nums:
                return nums[0]
    nums = re.findall(r"[-+]?\d*\.?\d+", text)
    return nums[-1] if nums else ""

def clean_response(text: str) -> str:
    """Обрезает ответ при появлении нового вопроса или маркера."""
    markers = ["Вопрос:", "Сколько будет", "Переведи", "?"]
    for marker in markers:
        idx = text.find(marker)
        if idx != -1 and idx > 0:
            return text[:idx].strip()
    return text.strip()

def extract_final_answer(text: str) -> str:
    """
    Извлекает финальный ответ из текста с рассуждениями.
    Ищет фразу 'Ответ:' или последнее число.
    """
    if "Ответ:" in text:
        parts = text.split("Ответ:")
        if len(parts) > 1:
            candidate = parts[-1].strip()
            nums = re.findall(r"[-+]?\d*\.?\d+", candidate)
            if nums:
                return nums[-1]
            return candidate
    return extract_number(text)

# ----------------------------------------------------------------
# Демонстрации
# ----------------------------------------------------------------
# Для перевода (одинаковы для всех режимов)
DEMO_TRANSLATIONS = [
    ("Переведи на английский: Привет, как дела?", "Hello, how are you?"),
    ("Переведи на английский: Сегодня отличная погода.", "Today is great weather."),
]

# Для математики (без CoT) – простые задачи
DEMO_MATH = [
    ("Сколько будет 2 + 2? Ответ дай числом.", "4"),
    ("Сколько будет 5 * 3? Ответ дай числом.", "15"),
]

# Для CoT – сложные многошаговые выражения с рассуждениями
DEMO_MATH_COT = [
    (
        "Вычисли значение выражения: (-80)-3*((-15)-4*(2-3*(12-5*(6-9)))). Ответ дай числом.",
        "Рассуждение:\n"
        "1. Внутренняя скобка: 6-9 = -3\n"
        "2. 5*(-3) = -15\n"
        "3. 12 - (-15) = 27\n"
        "4. 3*(27) = 81\n"
        "5. 2 - 81 = -79\n"
        "6. 4*(-79) = -316\n"
        "7. -15 - (-316) = 301\n"
        "8. 3*(301) = 903\n"
        "9. -80 - 903 = -983\n"
        "Ответ: -983."
    ),
    (
        "Вычисли значение выражения: (-210)+5*((-8)-2*(-10-3*(7-2*(4-11)))). Ответ дай числом.",
        "Рассуждение:\n"
        "1. Внутренняя скобка: 4-11 = -7\n"
        "2. 2*(-7) = -14\n"
        "3. 7 - (-14) = 21\n"
        "4. 3*(21) = 63\n"
        "5. -10 - 63 = -73\n"
        "6. 2*(-73) = -146\n"
        "7. -8 - (-146) = 138\n"
        "8. 5*138 = 690\n"
        "9. -210 + 690 = 480\n"
        "Ответ: 480."
    ),
]

# ----------------------------------------------------------------
# Тестовый датасет (включая сложное выражение)
# ----------------------------------------------------------------
test_dataset = [
    {"type": "translation", "input": "Переведи на английский: Я люблю программирование.", "target": "I love programming."},
    {"type": "translation", "input": "Переведи на английский: Который час?", "target": "What time is it?"},
    {"type": "translation", "input": "Переведи на английский: Это очень интересно.", "target": "This is very interesting."},
    {"type": "math", "input": "Сколько будет 10 - 4? Ответ дай числом.", "target": "6"},
    {"type": "math", "input": "Сколько будет 12 / 3? Ответ дай числом.", "target": "4"},
    {"type": "math", "input": "Сколько будет 7 + 8? Ответ дай числом.", "target": "15"},
    # Сложное выражение (аналогичное демонстрациям CoT)
    {"type": "math", "input": "Вычисли значение выражения: 100-6*((-4)-3*(5-2*(9-4*(1-6)))). Ответ дай числом.", "target": "-830"},
]

print("\n" + "=" * 70)
print("ТЕСТОВЫЙ ДАТАСЕТ")
print("=" * 70)
for i, ex in enumerate(test_dataset, 1):
    print(f"{i}. [{ex['type']}] {ex['input']} -> {ex['target']}")
print("=" * 70)

# ----------------------------------------------------------------
# Базовые функции генерации (без CoT)
# ----------------------------------------------------------------
def build_messages(demonstrations, question, system_msg=None):
    messages = []
    if system_msg and USE_SYSTEM:
        messages.append({"role": "system", "content": system_msg})
    for inp, out in demonstrations:
        messages.append({"role": "user", "content": inp})
        messages.append({"role": "assistant", "content": out})
    messages.append({"role": "user", "content": question})
    return messages

def generate_with_demos(question, demonstrations, system_msg=None,
                        max_new_tokens=MAX_NEW_TOKENS, temperature=0.2):
    messages = build_messages(demonstrations, question, system_msg)
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    start_time = time.time()
    with torch.inference_mode():
        result = generator(formatted, return_full_text=False, generation_config=gen_config)
    end_time = time.time()
    raw = result[0]["generated_text"].strip()
    cleaned = clean_response(raw)
    if not cleaned:
        cleaned = raw
    return cleaned, end_time - start_time, formatted

# ----------------------------------------------------------------
# Генерация с CoT (пошаговое рассуждение)
# ----------------------------------------------------------------
def generate_cot(question, demonstrations, system_msg=None, max_new_tokens=MAX_NEW_TOKENS*2):
    messages = build_messages(demonstrations, question, system_msg)
    # Добавляем инструкцию к вопросу, если её нет
    if "шаг за шагом" not in question.lower():
        modified_question = question + " Давайте подумаем шаг за шагом."
        messages[-1]["content"] = modified_question
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.3,
        top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    start_time = time.time()
    with torch.inference_mode():
        result = generator(formatted, return_full_text=False, generation_config=gen_config)
    end_time = time.time()
    raw = result[0]["generated_text"].strip()
    # Не обрезаем по маркерам, чтобы не потерять рассуждения
    cleaned = clean_response(raw)
    if not cleaned:
        cleaned = raw
    return cleaned, end_time - start_time, formatted

# ----------------------------------------------------------------
# Self-Consistency (голосование)
# ----------------------------------------------------------------
def generate_consensus(question, demonstrations, system_msg=None,
                       n=NUM_CONSISTENCY, temperature=0.7):
    candidates = []
    for _ in range(n):
        answer, _, _ = generate_with_demos(
            question, demonstrations, system_msg,
            max_new_tokens=MAX_NEW_TOKENS, temperature=temperature
        )
        # Извлекаем число для математических задач
        if "сколько" in question.lower() or "вычисли" in question.lower() or "Ответ дай числом" in question:
            final = extract_number(answer)
        else:
            final = answer
        candidates.append(final)
    counter = Counter(candidates)
    most_common = counter.most_common(1)[0][0]
    return most_common, candidates

# ----------------------------------------------------------------
# Загрузка метрик
# ----------------------------------------------------------------
bleu_metric = load("bleu") if HAS_EVALUATE else None
rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True) if HAS_ROUGE else None

# ----------------------------------------------------------------
# Функция оценки для заданного режима
# ----------------------------------------------------------------
def evaluate_mode(mode: str):
    print("\n" + "=" * 70)
    print(f"ОЦЕНКА В РЕЖИМЕ: {mode.upper()}")
    print("=" * 70)

    predictions = []
    refs = []
    perf_times = []
    perf_tokens = []
    math_correct = 0
    math_total = 0
    bleu_scores = []
    rouge_scores = []

    system_msg = "Ты — полезный ассистент. Отвечай кратко и только на последний вопрос. Не задавай новых вопросов."

    for idx, ex in enumerate(test_dataset, 1):
        ex_type = ex["type"]
        question = ex["input"]
        target = ex["target"]

        print(f"\n--- Пример {idx} ({ex_type}) ---")
        print(f"Вопрос: {question}")
        print(f"Эталон: {target}")

        if mode == "fewshot":
            demos = DEMO_TRANSLATIONS if ex_type == "translation" else DEMO_MATH
            answer, gen_time, _ = generate_with_demos(
                question, demos, system_msg, temperature=0.2
            )
        elif mode == "cot":
            if ex_type == "translation":
                demos = DEMO_TRANSLATIONS
                answer, gen_time, _ = generate_with_demos(
                    question, demos, system_msg, temperature=0.2
                )
            else:
                demos = DEMO_MATH_COT
                answer, gen_time, _ = generate_cot(
                    question, demos, system_msg
                )
        elif mode == "selfconsistency":
            demos = DEMO_TRANSLATIONS if ex_type == "translation" else DEMO_MATH
            if ex_type == "translation":
                answer, gen_time, _ = generate_with_demos(
                    question, demos, system_msg, temperature=0.2
                )
            else:
                answer, candidates = generate_consensus(
                    question, demos, system_msg, n=NUM_CONSISTENCY, temperature=0.7
                )
                # Оцениваем время как среднее по n генерациям (приблизительно)
                _, gen_time, _ = generate_with_demos(
                    question, demos, system_msg, temperature=0.7
                )
                gen_time *= NUM_CONSISTENCY

        # Очистка ответа (для CoT не обрезаем полностью)
        if mode != "cot":
            answer = clean_response(answer)

        print(f"Ответ модели: {answer}")
        print(f"   ⏱ {gen_time:.3f} сек")

        predictions.append(answer)
        refs.append([target])
        perf_times.append(gen_time)
        perf_tokens.append(len(tokenizer.encode(answer)))

        # Метрики качества
        if ex_type == "translation":
            if bleu_metric:
                try:
                    score = bleu_metric.compute(predictions=[answer], references=[[target]])
                    bleu_scores.append(score["bleu"])
                except Exception as e:
                      print(f"BLEU compute error: {e}")

            if rouge_scorer_obj:
                try:
                    rouge_scores.append(rouge_scorer_obj.score(target, answer)['rougeL'].fmeasure)
                except:
                    pass
        elif ex_type == "math":
            math_total += 1
            if mode == "selfconsistency":
                pred_num = answer  # уже извлечено число
            else:
                pred_num = extract_number(answer) if mode != "cot" else extract_final_answer(answer)
            ref_num = extract_number(target)
            if pred_num and ref_num and pred_num == ref_num:
                math_correct += 1
                print("   ✅ Математика верно")
            else:
                print(f"   ❌ Математика: извлечено '{pred_num}', ожидалось '{ref_num}'")

    # Итоговые метрики
    print("\n" + "=" * 70)
    print(f"РЕЗУЛЬТАТЫ КАЧЕСТВА ({mode.upper()})")
    print("=" * 70)
    if bleu_scores:
        print(f"📊 Средний BLEU: {sum(bleu_scores)/len(bleu_scores):.3f}")
    else:
        print("📊 BLEU: не доступен")
    if rouge_scores:
        print(f"📊 Средний ROUGE-L: {sum(rouge_scores)/len(rouge_scores):.3f}")
    if math_total:
        print(f"📊 Точность на математике: {math_correct/math_total:.2%} ({math_correct}/{math_total})")

    print("\n" + "=" * 70)
    print(f"СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ ({mode.upper()})")
    print("=" * 70)
    if perf_times:
        avg_time = sum(perf_times)/len(perf_times)
        avg_tokens = sum(perf_tokens)/len(perf_tokens)
        avg_tps = avg_tokens / avg_time if avg_time > 0 else 0
        print(f"Среднее время генерации: {avg_time:.3f} сек.")
        print(f"Среднее число токенов в ответе: {avg_tokens:.1f}")
        print(f"Средняя скорость: {avg_tps:.2f} токен/сек.")

    return {
        "mode": mode,
        "predictions": predictions,
        "bleu": sum(bleu_scores)/len(bleu_scores) if bleu_scores else None,
        "rouge": sum(rouge_scores)/len(rouge_scores) if rouge_scores else None,
        "math_accuracy": math_correct/math_total if math_total else None,
        "avg_time": sum(perf_times)/len(perf_times) if perf_times else None,
    }

# ----------------------------------------------------------------
# Запуск всех режимов и сравнение
# ----------------------------------------------------------------
if __name__ == "__main__":
    modes = ["fewshot", "cot", "selfconsistency"]
    results = {}
    for mode in modes:
        results[mode] = evaluate_mode(mode)

    # Сравнительная таблица
    print("\n" + "=" * 70)
    print("СРАВНЕНИЕ РЕЖИМОВ")
    print("=" * 70)
    print(f"{'Режим':<18} {'BLEU':<8} {'ROUGE-L':<10} {'Math Acc':<10} {'Время, с':<10}")
    for mode, res in results.items():
        bleu = f"{res['bleu']:.3f}" if res['bleu'] is not None else "—"
        rouge = f"{res['rouge']:.3f}" if res['rouge'] is not None else "—"
        acc = f"{res['math_accuracy']:.2%}" if res['math_accuracy'] is not None else "—"
        t = f"{res['avg_time']:.3f}" if res['avg_time'] is not None else "—"
        print(f"{mode:<18} {bleu:<8} {rouge:<10} {acc:<10} {t:<10}")

    print("\n" + "=" * 70)
    print("✅ ВСЕ РЕЖИМЫ ОЦЕНЕНЫ")
    print("=" * 70)

# Lora

In [ ]:
!pip install evaluate

In [ ]:
# ================================================================
# ЧИСТЫЙ LoRA Fine-Tuning
# Qwen/Qwen2.5-1.5B-Instruct
# ================================================================
# 1. УСТАНОВКА
# ================================================================

!pip install -q -U \
    transformers \
    peft \
    accelerate \
    datasets \
    evaluate \
    sacrebleu \
    matplotlib \
    "numpy<2.1"

# ================================================================
# 2. ИМПОРТЫ
# ================================================================

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)
import evaluate

# ================================================================
# 3. ПРОВЕРКА ОКРУЖЕНИЯ
# ================================================================

print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        print("Dtype: torch.bfloat16")
    else:
        DTYPE = torch.float16
        print("Dtype: torch.float16")
else:
    DTYPE = torch.float32
    print("Dtype: torch.float32")

print("=" * 70)

# ================================================================
# 4. КОНФИГУРАЦИЯ
# ================================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

OUTPUT_DIR = "./qwen25_1.5b_lora_translation"

MAX_LENGTH = 256

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training
NUM_EPOCHS = 20
LEARNING_RATE = 2e-4

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2

GRADIENT_ACCUMULATION_STEPS = 4

SEED = 42

# Early stopping
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# ================================================================
# 5. ДАТАСЕТ
# ================================================================

train_data = [
    {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
    {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
    {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
    {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
    {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
    {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
    {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
    {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
]

eval_data = [
    {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
    {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print()
print("=" * 70)
print("DATASET")
print("=" * 70)

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))

print()
print("Пример:")
print(train_dataset[0])

print("=" * 70)

# ================================================================
# 6. TOKENIZER
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocab size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

print("=" * 70)

# ================================================================
# 7. ФОРМИРОВАНИЕ PROMPT
# ================================================================

def build_messages(example):
    return [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]

def tokenize_example(example):
    messages = build_messages(example)
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    prompt_messages = [{"role": "user", "content": example["instruction"]}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    prompt_tokens = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    prompt_length = len(prompt_tokens["input_ids"])

    labels = input_ids.copy()
    for i in range(min(prompt_length, len(labels))):
        labels[i] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# ================================================================
# 8. TOKENIZATION
# ================================================================

print()
print("=" * 70)
print("TOKENIZATION")
print("=" * 70)

tokenized_train = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)

tokenized_eval = eval_dataset.map(
    tokenize_example,
    remove_columns=eval_dataset.column_names,
)

print("Train examples:", len(tokenized_train))
print("Eval examples :", len(tokenized_eval))

print("=" * 70)

# ================================================================
# 9. ЗАГРУЗКА БАЗОВОЙ МОДЕЛИ
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА BASE MODEL")
print("=" * 70)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

print("Model loaded.")
print("=" * 70)

# ================================================================
# 10. LoRA CONFIG
# ================================================================

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# ================================================================
# 11. ДОБАВЛЯЕМ LoRA
# ================================================================

model = get_peft_model(model, lora_config)

# ================================================================
# 12. ПРОВЕРКА ПАРАМЕТРОВ
# ================================================================

print()
print("=" * 70)
print("LoRA PARAMETER CHECK")
print("=" * 70)

model.print_trainable_parameters()

print("=" * 70)

# ================================================================
# 13. ПРОВЕРКА: ТОЛЬКО LoRA ОБУЧАЕТСЯ
# ================================================================

trainable_names = []
for name, param in model.named_parameters():
    if param.requires_grad:
        trainable_names.append(name)

print()
print("Количество trainable tensors:", len(trainable_names))
print()
print("Первые trainable parameters:")
for name in trainable_names[:20]:
    print("  ", name)

# ================================================================
# 14. DATA COLLATOR
# ================================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

# ================================================================
# 15. TRAINING ARGUMENTS (с warmup_steps)
# ================================================================

total_steps = (len(tokenized_train) // (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
warmup_steps = int(0.05 * total_steps)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    optim="adamw_torch",
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=(torch.cuda.is_available() and DTYPE == torch.float16),
    bf16=(torch.cuda.is_available() and DTYPE == torch.bfloat16),
    gradient_checkpointing=True,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

# ================================================================
# 16. МЕТРИКА BLEU (исправленная)
# ================================================================

bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    """
    Вычисляет BLEU на валидационном наборе.
    Принимает логиты и метки, делает argmax и декодирует.
    """
    logits, labels = eval_pred
    # logits: (batch_size, seq_len, vocab_size)
    predictions = np.argmax(logits, axis=-1)  # (batch_size, seq_len)

    # Заменяем -100 на pad_token_id для декодирования
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Декодируем
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Проверяем, что количество совпадает (на случай, если батчи не объединились)
    if len(decoded_preds) != len(decoded_labels):
        # Если не совпадает, возможно, один из них пустой – выравниваем
        min_len = min(len(decoded_preds), len(decoded_labels))
        decoded_preds = decoded_preds[:min_len]
        decoded_labels = decoded_labels[:min_len]

    # Для sacrebleu каждый референс должен быть списком
    decoded_labels = [[ref] for ref in decoded_labels]

    result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

# ================================================================
# 17. TRAINER (без preprocess_logits_for_metrics для надёжности)
# ================================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

# ================================================================
# 18. ТРЕНИРОВКА
# ================================================================

print()
print("=" * 70)
print("НАЧАЛО LoRA TRAINING")
print("=" * 70)

train_result = trainer.train()

print()
print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

print(train_result)

# ================================================================
# 19. СОХРАНЕНИЕ LoRA
# ================================================================

print()
print("=" * 70)
print("СОХРАНЕНИЕ LoRA")
print("=" * 70)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("LoRA adapter сохранён:")
print(OUTPUT_DIR)

print("=" * 70)

# ================================================================
# 20. ПРОВЕРКА ФАЙЛОВ
# ================================================================

print()
print("Содержимое директории:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, filename)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"{filename:40s}{size_mb:10.2f} MB")

# ================================================================
# 21. ГРАФИК ОБУЧЕНИЯ
# ================================================================

print()
print("=" * 70)
print("ГРАФИК ОБУЧЕНИЯ")
print("=" * 70)

log_history = trainer.state.log_history

train_losses = []
eval_losses = []
bleu_scores = []
epochs = []

for log in log_history:
    if "loss" in log and "epoch" in log:
        train_losses.append(log["loss"])
        epochs.append(log["epoch"])
    if "eval_loss" in log:
        eval_losses.append((log["epoch"], log["eval_loss"]))
    if "bleu" in log:
        bleu_scores.append((log["epoch"], log["bleu"]))

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss", color="tab:red")
ax1.plot(epochs, train_losses, label="Train Loss", color="tab:red", marker="o")
if eval_losses:
    eval_epochs, eval_vals = zip(*eval_losses)
    ax1.plot(eval_epochs, eval_vals, label="Eval Loss", color="tab:orange", marker="s")
ax1.tick_params(axis="y", labelcolor="tab:red")
ax1.legend(loc="upper left")

if bleu_scores:
    ax2 = ax1.twinx()
    ax2.set_ylabel("BLEU", color="tab:blue")
    bleu_epochs, bleu_vals = zip(*bleu_scores)
    ax2.plot(bleu_epochs, bleu_vals, label="BLEU", color="tab:blue", marker="^")
    ax2.tick_params(axis="y", labelcolor="tab:blue")
    ax2.legend(loc="upper right")

plt.title("Training and Validation Metrics")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_plot.png"), dpi=150)
plt.show()

print(f"График сохранён в {os.path.join(OUTPUT_DIR, 'training_plot.png')}")
print("=" * 70)

# ================================================================
# 22. ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ
# ================================================================

print()
print("=" * 70)
print("ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ")
print("=" * 70)

# Определим функцию инференса (она понадобится ниже)
def generate_translation(
    instruction,
    max_new_tokens=64,
    temperature=0.2,
):
    model.eval()
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return answer.strip()

references = [ex["output"] for ex in eval_data]
predictions = []
for ex in eval_data:
    pred = generate_translation(ex["instruction"])
    predictions.append(pred)

bleu_score = bleu_metric.compute(predictions=predictions, references=[[ref] for ref in references])
print(f"BLEU на валидационном наборе: {bleu_score['score']:.2f}")

for i, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"\nПример {i+1}:")
    print(f"  Инструкция: {eval_data[i]['instruction']}")
    print(f"  Ожидалось : {ref}")
    print(f"  Получено  : {pred}")

print("=" * 70)

# ================================================================
# 23. ТЕСТ НА TRAIN EXAMPLE
# ================================================================

print()
print("=" * 70)
print("TEST: TRAIN EXAMPLE")
print("=" * 70)

test_instruction = "Переведи на английский: Привет, как дела?"
answer = generate_translation(test_instruction)

print("INPUT :")
print(test_instruction)
print()
print("OUTPUT:")
print(answer)
print("=" * 70)

# ================================================================
# 24. ТЕСТ НА НОВЫХ ПРИМЕРАХ
# ================================================================

test_examples = [
    "Переведи на английский: Я хочу пить.",
    "Переведи на английский: Где находится библиотека?",
    "Переведи на английский: Я люблю Python.",
    "Переведи на английский: До свидания!",
    "Переведи на английский: Как тебя зовут?",
]

print()
print("=" * 70)
print("TEST: NEW EXAMPLES")
print("=" * 70)

for instruction in test_examples:
    answer = generate_translation(instruction)
    print()
    print("INPUT :", instruction)
    print("OUTPUT:", answer)

print("=" * 70)

In [ ]:
# ================================================================
# ЧИСТЫЙ LoRA Fine-Tuning
# TinyLlama/TinyLlama-1.1B-Chat-v1.0
# ================================================================
# 1. УСТАНОВКА
# ================================================================

!pip install -q -U \
    transformers \
    peft \
    accelerate \
    datasets \
    evaluate \
    sacrebleu \
    matplotlib \
    "numpy<2.1"

# ================================================================
# 2. ИМПОРТЫ
# ================================================================

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)
import evaluate

# ================================================================
# 3. ПРОВЕРКА ОКРУЖЕНИЯ
# ================================================================

print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        print("Dtype: torch.bfloat16")
    else:
        DTYPE = torch.float16
        print("Dtype: torch.float16")
else:
    DTYPE = torch.float32
    print("Dtype: torch.float32")

print("=" * 70)

# ================================================================
# 4. КОНФИГУРАЦИЯ (АДАПТИРОВАНА ДЛЯ TinyLlama-1.1B)
# ================================================================

# <-- ИЗМЕНЕНО: новая модель
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

OUTPUT_DIR = "./tinyllama_1.1b_lora_translation"

MAX_LENGTH = 256

# LoRA
LORA_R = 16                     # можно оставить 16 или попробовать 32
LORA_ALPHA = 32                 # 2 * r
LORA_DROPOUT = 0.05

# Training
NUM_EPOCHS = 20                 # с Early Stopping можно оставить 20
LEARNING_RATE = 2e-4            # стандарт для LoRA

TRAIN_BATCH_SIZE = 2            # для 1.1B модели можно попробовать 4, если память позволяет
EVAL_BATCH_SIZE = 2

GRADIENT_ACCUMULATION_STEPS = 4

SEED = 42

# Early stopping
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# ================================================================
# 5. ДАТАСЕТ
# ================================================================

train_data = [
    {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
    {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
    {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
    {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
    {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
    {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
    {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
    {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
]

eval_data = [
    {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
    {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print()
print("=" * 70)
print("DATASET")
print("=" * 70)

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))

print()
print("Пример:")
print(train_dataset[0])

print("=" * 70)

# ================================================================
# 6. TOKENIZER
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,      # <-- ИЗМЕНЕНО: для TinyLlama trust_remote_code обычно не требуется, но оставим
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocab size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

print("=" * 70)

# ================================================================
# 7. ФОРМИРОВАНИЕ PROMPT
# ================================================================

def build_messages(example):
    return [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]

def tokenize_example(example):
    messages = build_messages(example)
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    prompt_messages = [{"role": "user", "content": example["instruction"]}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    prompt_tokens = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    prompt_length = len(prompt_tokens["input_ids"])

    labels = input_ids.copy()
    for i in range(min(prompt_length, len(labels))):
        labels[i] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# ================================================================
# 8. TOKENIZATION
# ================================================================

print()
print("=" * 70)
print("TOKENIZATION")
print("=" * 70)

tokenized_train = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)

tokenized_eval = eval_dataset.map(
    tokenize_example,
    remove_columns=eval_dataset.column_names,
)

print("Train examples:", len(tokenized_train))
print("Eval examples :", len(tokenized_eval))

print("=" * 70)

# ================================================================
# 9. ЗАГРУЗКА БАЗОВОЙ МОДЕЛИ
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА BASE MODEL")
print("=" * 70)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,      # <-- можно убрать, если не требуется
)

model.config.use_cache = False

print("Model loaded.")
print("=" * 70)

# ================================================================
# 10. LoRA CONFIG
# ================================================================

# <-- ИЗМЕНЕНО: target_modules для TinyLlama (структура как у LLaMA)
# Можно оставить все четыре проекции, но для экономии памяти можно ограничиться ["q_proj","v_proj"]
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # все проекции внимания
)

# ================================================================
# 11. ДОБАВЛЯЕМ LoRA
# ================================================================

model = get_peft_model(model, lora_config)

# ================================================================
# 12. ПРОВЕРКА ПАРАМЕТРОВ
# ================================================================

print()
print("=" * 70)
print("LoRA PARAMETER CHECK")
print("=" * 70)

model.print_trainable_parameters()

print("=" * 70)

# ================================================================
# 13. ПРОВЕРКА: ТОЛЬКО LoRA ОБУЧАЕТСЯ
# ================================================================

trainable_names = []
for name, param in model.named_parameters():
    if param.requires_grad:
        trainable_names.append(name)

print()
print("Количество trainable tensors:", len(trainable_names))
print()
print("Первые trainable parameters:")
for name in trainable_names[:20]:
    print("  ", name)

# ================================================================
# 14. DATA COLLATOR
# ================================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

# ================================================================
# 15. TRAINING ARGUMENTS (с warmup_steps)
# ================================================================

total_steps = (len(tokenized_train) // (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
warmup_steps = int(0.05 * total_steps)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    optim="adamw_torch",
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=(torch.cuda.is_available() and DTYPE == torch.float16),
    bf16=(torch.cuda.is_available() and DTYPE == torch.bfloat16),
    gradient_checkpointing=True,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

# ================================================================
# 16. МЕТРИКА BLEU (исправленная)
# ================================================================

bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    """
    Вычисляет BLEU на валидационном наборе.
    Принимает логиты и метки, делает argmax и декодирует.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    if len(decoded_preds) != len(decoded_labels):
        min_len = min(len(decoded_preds), len(decoded_labels))
        decoded_preds = decoded_preds[:min_len]
        decoded_labels = decoded_labels[:min_len]

    decoded_labels = [[ref] for ref in decoded_labels]

    result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

# ================================================================
# 17. TRAINER
# ================================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

# ================================================================
# 18. ТРЕНИРОВКА
# ================================================================

print()
print("=" * 70)
print("НАЧАЛО LoRA TRAINING")
print("=" * 70)

train_result = trainer.train()

print()
print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

print(train_result)

# ================================================================
# 19. СОХРАНЕНИЕ LoRA
# ================================================================

print()
print("=" * 70)
print("СОХРАНЕНИЕ LoRA")
print("=" * 70)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("LoRA adapter сохранён:")
print(OUTPUT_DIR)

print("=" * 70)

# ================================================================
# 20. ПРОВЕРКА ФАЙЛОВ
# ================================================================

print()
print("Содержимое директории:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, filename)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"{filename:40s}{size_mb:10.2f} MB")

# ================================================================
# 21. ГРАФИК ОБУЧЕНИЯ
# ================================================================

print()
print("=" * 70)
print("ГРАФИК ОБУЧЕНИЯ")
print("=" * 70)

log_history = trainer.state.log_history

train_losses = []
eval_losses = []
bleu_scores = []
epochs = []

for log in log_history:
    if "loss" in log and "epoch" in log:
        train_losses.append(log["loss"])
        epochs.append(log["epoch"])
    if "eval_loss" in log:
        eval_losses.append((log["epoch"], log["eval_loss"]))
    if "bleu" in log:
        bleu_scores.append((log["epoch"], log["bleu"]))

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss", color="tab:red")
ax1.plot(epochs, train_losses, label="Train Loss", color="tab:red", marker="o")
if eval_losses:
    eval_epochs, eval_vals = zip(*eval_losses)
    ax1.plot(eval_epochs, eval_vals, label="Eval Loss", color="tab:orange", marker="s")
ax1.tick_params(axis="y", labelcolor="tab:red")
ax1.legend(loc="upper left")

if bleu_scores:
    ax2 = ax1.twinx()
    ax2.set_ylabel("BLEU", color="tab:blue")
    bleu_epochs, bleu_vals = zip(*bleu_scores)
    ax2.plot(bleu_epochs, bleu_vals, label="BLEU", color="tab:blue", marker="^")
    ax2.tick_params(axis="y", labelcolor="tab:blue")
    ax2.legend(loc="upper right")

plt.title("Training and Validation Metrics")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_plot.png"), dpi=150)
plt.show()

print(f"График сохранён в {os.path.join(OUTPUT_DIR, 'training_plot.png')}")
print("=" * 70)

# ================================================================
# 22. ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ
# ================================================================

print()
print("=" * 70)
print("ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ")
print("=" * 70)

def generate_translation(
    instruction,
    max_new_tokens=64,
    temperature=0.2,
):
    model.eval()
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return answer.strip()

references = [ex["output"] for ex in eval_data]
predictions = []
for ex in eval_data:
    pred = generate_translation(ex["instruction"])
    predictions.append(pred)

bleu_score = bleu_metric.compute(predictions=predictions, references=[[ref] for ref in references])
print(f"BLEU на валидационном наборе: {bleu_score['score']:.2f}")

for i, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"\nПример {i+1}:")
    print(f"  Инструкция: {eval_data[i]['instruction']}")
    print(f"  Ожидалось : {ref}")
    print(f"  Получено  : {pred}")

print("=" * 70)

# ================================================================
# 23. ТЕСТ НА TRAIN EXAMPLE
# ================================================================

print()
print("=" * 70)
print("TEST: TRAIN EXAMPLE")
print("=" * 70)

test_instruction = "Переведи на английский: Привет, как дела?"
answer = generate_translation(test_instruction)

print("INPUT :")
print(test_instruction)
print()
print("OUTPUT:")
print(answer)
print("=" * 70)

# ================================================================
# 24. ТЕСТ НА НОВЫХ ПРИМЕРАХ
# ================================================================

test_examples = [
    "Переведи на английский: Я хочу пить.",
    "Переведи на английский: Где находится библиотека?",
    "Переведи на английский: Я люблю Python.",
    "Переведи на английский: До свидания!",
    "Переведи на английский: Как тебя зовут?",
]

print()
print("=" * 70)
print("TEST: NEW EXAMPLES")
print("=" * 70)

for instruction in test_examples:
    answer = generate_translation(instruction)
    print()
    print("INPUT :", instruction)
    print("OUTPUT:", answer)

print("=" * 70)

In [ ]:
# ================================================================
# ЧИСТЫЙ LoRA Fine-Tuning
# GPT-2 (базовая модель)
# ================================================================
# 1. УСТАНОВКА
# ================================================================

!pip install -q -U \
    transformers \
    peft \
    accelerate \
    datasets \
    evaluate \
    sacrebleu \
    matplotlib \
    "numpy<2.1"

# ================================================================
# 2. ИМПОРТЫ
# ================================================================

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)
import evaluate

# ================================================================
# 3. ПРОВЕРКА ОКРУЖЕНИЯ
# ================================================================

print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        print("Dtype: torch.bfloat16")
    else:
        DTYPE = torch.float16
        print("Dtype: torch.float16")
else:
    DTYPE = torch.float32
    print("Dtype: torch.float32")

print("=" * 70)

# ================================================================
# 4. КОНФИГУРАЦИЯ (АДАПТИРОВАНА ДЛЯ GPT-2)
# ================================================================

# <-- ИЗМЕНЕНО: модель GPT-2 (можно взять gpt2-medium, gpt2-large)
MODEL_NAME = "gpt2"  # или "gpt2-medium", "gpt2-large"

OUTPUT_DIR = "./gpt2_lora_translation"

MAX_LENGTH = 256

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training
NUM_EPOCHS = 20
LEARNING_RATE = 2e-4

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2

GRADIENT_ACCUMULATION_STEPS = 4

SEED = 42

# Early stopping
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# ================================================================
# 5. ДАТАСЕТ
# ================================================================

train_data = [
    {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
    {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
    {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
    {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
    {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
    {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
    {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
    {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
]

eval_data = [
    {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
    {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print()
print("=" * 70)
print("DATASET")
print("=" * 70)

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))

print()
print("Пример:")
print(train_dataset[0])

print("=" * 70)

# ================================================================
# 6. TOKENIZER
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

# <-- ИЗМЕНЕНО: для GPT-2 нужно явно установить pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocab size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

print("=" * 70)

# ================================================================
# 7. ФОРМИРОВАНИЕ PROMPT (РУЧНОЕ, БЕЗ CHAT_TEMPLATE)
# ================================================================

# <-- ИЗМЕНЕНО: GPT-2 не поддерживает apply_chat_template,
# поэтому формируем промпт вручную

def build_prompt(instruction, output=None):
    """
    Формирует промпт для GPT-2.
    Если output=None — только инструкция (для инференса).
    """
    prompt = f"Инструкция: {instruction}\nОтвет:"
    if output is not None:
        prompt += f" {output}"
    return prompt

def tokenize_example(example):
    # Полный текст (инструкция + ответ)
    full_text = build_prompt(example["instruction"], example["output"])
    # Только инструкция (для вычисления длины промпта)
    prompt_text = build_prompt(example["instruction"], None)

    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    prompt_tokens = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    prompt_length = len(prompt_tokens["input_ids"])

    labels = input_ids.copy()
    for i in range(min(prompt_length, len(labels))):
        labels[i] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# ================================================================
# 8. TOKENIZATION
# ================================================================

print()
print("=" * 70)
print("TOKENIZATION")
print("=" * 70)

tokenized_train = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)

tokenized_eval = eval_dataset.map(
    tokenize_example,
    remove_columns=eval_dataset.column_names,
)

print("Train examples:", len(tokenized_train))
print("Eval examples :", len(tokenized_eval))

print("=" * 70)

# ================================================================
# 9. ЗАГРУЗКА БАЗОВОЙ МОДЕЛИ
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА BASE MODEL")
print("=" * 70)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
)

model.config.use_cache = False

print("Model loaded.")
print("=" * 70)

# ================================================================
# 10. LoRA CONFIG (АДАПТИРОВАНА ДЛЯ GPT-2)
# ================================================================

# <-- ИЗМЕНЕНО: target_modules для GPT-2
# У GPT-2 в self-attention одна матрица c_attn (QKV вместе) и c_proj (выходная)
# Также можно добавить MLP-слои: c_fc, c_proj
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn", "c_proj"],  # для GPT-2
)

# ================================================================
# 11. ДОБАВЛЯЕМ LoRA
# ================================================================

model = get_peft_model(model, lora_config)

# ================================================================
# 12. ПРОВЕРКА ПАРАМЕТРОВ
# ================================================================

print()
print("=" * 70)
print("LoRA PARAMETER CHECK")
print("=" * 70)

model.print_trainable_parameters()

print("=" * 70)

# ================================================================
# 13. ПРОВЕРКА: ТОЛЬКО LoRA ОБУЧАЕТСЯ
# ================================================================

trainable_names = []
for name, param in model.named_parameters():
    if param.requires_grad:
        trainable_names.append(name)

print()
print("Количество trainable tensors:", len(trainable_names))
print()
print("Первые trainable parameters:")
for name in trainable_names[:20]:
    print("  ", name)

# ================================================================
# 14. DATA COLLATOR
# ================================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

# ================================================================
# 15. TRAINING ARGUMENTS
# ================================================================

total_steps = (len(tokenized_train) // (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
warmup_steps = int(0.05 * total_steps)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    optim="adamw_torch",
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=(torch.cuda.is_available() and DTYPE == torch.float16),
    bf16=(torch.cuda.is_available() and DTYPE == torch.bfloat16),
    gradient_checkpointing=True,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

# ================================================================
# 16. МЕТРИКА BLEU
# ================================================================

bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    if len(decoded_preds) != len(decoded_labels):
        min_len = min(len(decoded_preds), len(decoded_labels))
        decoded_preds = decoded_preds[:min_len]
        decoded_labels = decoded_labels[:min_len]

    decoded_labels = [[ref] for ref in decoded_labels]

    result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

# ================================================================
# 17. TRAINER
# ================================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

# ================================================================
# 18. ТРЕНИРОВКА
# ================================================================

print()
print("=" * 70)
print("НАЧАЛО LoRA TRAINING")
print("=" * 70)

train_result = trainer.train()

print()
print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

print(train_result)

# ================================================================
# 19. СОХРАНЕНИЕ LoRA
# ================================================================

print()
print("=" * 70)
print("СОХРАНЕНИЕ LoRA")
print("=" * 70)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("LoRA adapter сохранён:")
print(OUTPUT_DIR)

print("=" * 70)

# ================================================================
# 20. ПРОВЕРКА ФАЙЛОВ
# ================================================================

print()
print("Содержимое директории:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, filename)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"{filename:40s}{size_mb:10.2f} MB")

# ================================================================
# 21. ГРАФИК ОБУЧЕНИЯ
# ================================================================

print()
print("=" * 70)
print("ГРАФИК ОБУЧЕНИЯ")
print("=" * 70)

log_history = trainer.state.log_history

train_losses = []
eval_losses = []
bleu_scores = []
epochs = []

for log in log_history:
    if "loss" in log and "epoch" in log:
        train_losses.append(log["loss"])
        epochs.append(log["epoch"])
    if "eval_loss" in log:
        eval_losses.append((log["epoch"], log["eval_loss"]))
    if "bleu" in log:
        bleu_scores.append((log["epoch"], log["bleu"]))

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss", color="tab:red")
ax1.plot(epochs, train_losses, label="Train Loss", color="tab:red", marker="o")
if eval_losses:
    eval_epochs, eval_vals = zip(*eval_losses)
    ax1.plot(eval_epochs, eval_vals, label="Eval Loss", color="tab:orange", marker="s")
ax1.tick_params(axis="y", labelcolor="tab:red")
ax1.legend(loc="upper left")

if bleu_scores:
    ax2 = ax1.twinx()
    ax2.set_ylabel("BLEU", color="tab:blue")
    bleu_epochs, bleu_vals = zip(*bleu_scores)
    ax2.plot(bleu_epochs, bleu_vals, label="BLEU", color="tab:blue", marker="^")
    ax2.tick_params(axis="y", labelcolor="tab:blue")
    ax2.legend(loc="upper right")

plt.title("Training and Validation Metrics")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_plot.png"), dpi=150)
plt.show()

print(f"График сохранён в {os.path.join(OUTPUT_DIR, 'training_plot.png')}")
print("=" * 70)

# ================================================================
# 22. ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ
# ================================================================

print()
print("=" * 70)
print("ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ")
print("=" * 70)

def generate_translation(
    instruction,
    max_new_tokens=64,
    temperature=0.2,
):
    model.eval()
    prompt = build_prompt(instruction, None)  # только инструкция
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return answer.strip()

references = [ex["output"] for ex in eval_data]
predictions = []
for ex in eval_data:
    pred = generate_translation(ex["instruction"])
    predictions.append(pred)

bleu_score = bleu_metric.compute(predictions=predictions, references=[[ref] for ref in references])
print(f"BLEU на валидационном наборе: {bleu_score['score']:.2f}")

for i, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"\nПример {i+1}:")
    print(f"  Инструкция: {eval_data[i]['instruction']}")
    print(f"  Ожидалось : {ref}")
    print(f"  Получено  : {pred}")

print("=" * 70)

# ================================================================
# 23. ТЕСТ НА TRAIN EXAMPLE
# ================================================================

print()
print("=" * 70)
print("TEST: TRAIN EXAMPLE")
print("=" * 70)

test_instruction = "Переведи на английский: Привет, как дела?"
answer = generate_translation(test_instruction)

print("INPUT :")
print(test_instruction)
print()
print("OUTPUT:")
print(answer)
print("=" * 70)

# ================================================================
# 24. ТЕСТ НА НОВЫХ ПРИМЕРАХ
# ================================================================

test_examples = [
    "Переведи на английский: Я хочу пить.",
    "Переведи на английский: Где находится библиотека?",
    "Переведи на английский: Я люблю Python.",
    "Переведи на английский: До свидания!",
    "Переведи на английский: Как тебя зовут?",
]

print()
print("=" * 70)
print("TEST: NEW EXAMPLES")
print("=" * 70)

for instruction in test_examples:
    answer = generate_translation(instruction)
    print()
    print("INPUT :", instruction)
    print("OUTPUT:", answer)

print("=" * 70)

# QLORA

In [ ]:
# ================================================================
# QLoRA Fine-Tuning (4-bit Quantized)
# TinyLlama/TinyLlama-1.1B-Chat-v1.0
# ================================================================
# 1. УСТАНОВКА
# ================================================================

# <-- ИЗМЕНЕНО: добавлена библиотека bitsandbytes
!pip install -q -U \
    transformers \
    peft \
    accelerate \
    datasets \
    evaluate \
    sacrebleu \
    matplotlib \
    bitsandbytes \
    "numpy<2.1"

# ================================================================
# 2. ИМПОРТЫ
# ================================================================

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    BitsAndBytesConfig,          # <-- НОВОЕ: для 4-битного квантования
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)
import evaluate

# ================================================================
# 3. ПРОВЕРКА ОКРУЖЕНИЯ
# ================================================================

print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        print("Dtype: torch.bfloat16")
    else:
        DTYPE = torch.float16
        print("Dtype: torch.float16")
else:
    DTYPE = torch.float32
    print("Dtype: torch.float32")

print("=" * 70)

# ================================================================
# 4. КОНФИГУРАЦИЯ (АДАПТИРОВАНА ДЛЯ QLoRA)
# ================================================================

# <-- ИЗМЕНЕНО: можно подставить любую модель
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# Альтернативы:
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

OUTPUT_DIR = "./tinyllama_1.1b_qlora_translation"   # <-- папка для QLoRA

MAX_LENGTH = 256

# LoRA (те же параметры, что и для обычного LoRA)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training
NUM_EPOCHS = 20
LEARNING_RATE = 2e-4             # QLoRA использует ту же LR

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2

GRADIENT_ACCUMULATION_STEPS = 4

SEED = 42

# Early stopping
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# ================================================================
# 5. ДАТАСЕТ
# ================================================================

train_data = [
    {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
    {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
    {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
    {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
    {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
    {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
    {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
    {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
]

eval_data = [
    {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
    {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print()
print("=" * 70)
print("DATASET")
print("=" * 70)

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))

print()
print("Пример:")
print(train_dataset[0])

print("=" * 70)

# ================================================================
# 6. TOKENIZER
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocab size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

print("=" * 70)

# ================================================================
# 7. ФОРМИРОВАНИЕ PROMPT
# ================================================================

def build_messages(example):
    return [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]

def tokenize_example(example):
    messages = build_messages(example)
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    prompt_messages = [{"role": "user", "content": example["instruction"]}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    prompt_tokens = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    prompt_length = len(prompt_tokens["input_ids"])

    labels = input_ids.copy()
    for i in range(min(prompt_length, len(labels))):
        labels[i] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# ================================================================
# 8. TOKENIZATION
# ================================================================

print()
print("=" * 70)
print("TOKENIZATION")
print("=" * 70)

tokenized_train = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)

tokenized_eval = eval_dataset.map(
    tokenize_example,
    remove_columns=eval_dataset.column_names,
)

print("Train examples:", len(tokenized_train))
print("Eval examples :", len(tokenized_eval))

print("=" * 70)

# ================================================================
# 9. КОНФИГУРАЦИЯ КВАНТОВАНИЯ (QLoRA)
# ================================================================

print()
print("=" * 70)
print("НАСТРОЙКА 4-BIT КВАНТОВАНИЯ")
print("=" * 70)

# <-- НОВОЕ: BitsAndBytesConfig для QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                         # Включаем 4-битное квантование
    bnb_4bit_quant_type="nf4",                 # NF4 — оптимальный для LLM
    bnb_4bit_use_double_quant=True,            # Double Quantization — экономия памяти
    bnb_4bit_compute_dtype=DTYPE,              # Деквантуем до bf16/fp16 для вычислений
)

print("4-bit квантование включено:")
print(f"  Тип: NF4")
print(f"  Double Quantization: True")
print(f"  Compute dtype: {DTYPE}")

print("=" * 70)

# ================================================================
# 10. ЗАГРУЗКА КВАНТОВАННОЙ МОДЕЛИ
# ================================================================

print()
print("=" * 70)
print("ЗАГРУЗКА КВАНТОВАННОЙ МОДЕЛИ (4-bit)")
print("=" * 70)

# <-- ИЗМЕНЕНО: добавлен quantization_config
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,            # <-- КЛЮЧЕВОЕ ОТЛИЧИЕ QLoRA
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

print("Квантованная модель загружена.")
print("=" * 70)

# ================================================================
# 11. LoRA CONFIG
# ================================================================

# <-- target_modules зависят от архитектуры модели.
# Для TinyLlama (как и LLaMA, Mistral, Qwen) используются:
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
# Если модель другая (например, GPT-2), нужно изменить на ["c_attn", "c_proj"]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

# ================================================================
# 12. ДОБАВЛЯЕМ LoRA
# ================================================================

model = get_peft_model(model, lora_config)

# ================================================================
# 13. ПРОВЕРКА ПАРАМЕТРОВ
# ================================================================

print()
print("=" * 70)
print("LoRA PARAMETER CHECK")
print("=" * 70)

model.print_trainable_parameters()

print("=" * 70)

# ================================================================
# 14. DATA COLLATOR
# ================================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

# ================================================================
# 15. TRAINING ARGUMENTS
# ================================================================

total_steps = (len(tokenized_train) // (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
warmup_steps = int(0.05 * total_steps)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    optim="adamw_torch",
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=(torch.cuda.is_available() and DTYPE == torch.float16),
    bf16=(torch.cuda.is_available() and DTYPE == torch.bfloat16),
    gradient_checkpointing=True,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

# ================================================================
# 16. МЕТРИКА BLEU
# ================================================================

bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    if len(decoded_preds) != len(decoded_labels):
        min_len = min(len(decoded_preds), len(decoded_labels))
        decoded_preds = decoded_preds[:min_len]
        decoded_labels = decoded_labels[:min_len]

    decoded_labels = [[ref] for ref in decoded_labels]

    result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

# ================================================================
# 17. TRAINER
# ================================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

# ================================================================
# 18. ТРЕНИРОВКА
# ================================================================

print()
print("=" * 70)
print("НАЧАЛО QLoRA TRAINING (4-bit)")
print("=" * 70)

train_result = trainer.train()

print()
print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

print(train_result)

# ================================================================
# 19. СОХРАНЕНИЕ LoRA
# ================================================================

print()
print("=" * 70)
print("СОХРАНЕНИЕ QLoRA АДАПТЕРА")
print("=" * 70)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("QLoRA adapter сохранён:")
print(OUTPUT_DIR)

print("=" * 70)

# ================================================================
# 20. ПРОВЕРКА ФАЙЛОВ
# ================================================================

print()
print("Содержимое директории:")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, filename)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"{filename:40s}{size_mb:10.2f} MB")

# ================================================================
# 21. ГРАФИК ОБУЧЕНИЯ
# ================================================================

print()
print("=" * 70)
print("ГРАФИК ОБУЧЕНИЯ")
print("=" * 70)

log_history = trainer.state.log_history

train_losses = []
eval_losses = []
bleu_scores = []
epochs = []

for log in log_history:
    if "loss" in log and "epoch" in log:
        train_losses.append(log["loss"])
        epochs.append(log["epoch"])
    if "eval_loss" in log:
        eval_losses.append((log["epoch"], log["eval_loss"]))
    if "bleu" in log:
        bleu_scores.append((log["epoch"], log["bleu"]))

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss", color="tab:red")
ax1.plot(epochs, train_losses, label="Train Loss", color="tab:red", marker="o")
if eval_losses:
    eval_epochs, eval_vals = zip(*eval_losses)
    ax1.plot(eval_epochs, eval_vals, label="Eval Loss", color="tab:orange", marker="s")
ax1.tick_params(axis="y", labelcolor="tab:red")
ax1.legend(loc="upper left")

if bleu_scores:
    ax2 = ax1.twinx()
    ax2.set_ylabel("BLEU", color="tab:blue")
    bleu_epochs, bleu_vals = zip(*bleu_scores)
    ax2.plot(bleu_epochs, bleu_vals, label="BLEU", color="tab:blue", marker="^")
    ax2.tick_params(axis="y", labelcolor="tab:blue")
    ax2.legend(loc="upper right")

plt.title("QLoRA Training and Validation Metrics (4-bit)")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_plot.png"), dpi=150)
plt.show()

print(f"График сохранён в {os.path.join(OUTPUT_DIR, 'training_plot.png')}")
print("=" * 70)

# ================================================================
# 22. ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ
# ================================================================

print()
print("=" * 70)
print("ФИНАЛЬНАЯ ОЦЕНКА BLEU НА ВАЛИДАЦИИ (QLoRA)")
print("=" * 70)

def generate_translation(
    instruction,
    max_new_tokens=64,
    temperature=0.2,
):
    model.eval()
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return answer.strip()

references = [ex["output"] for ex in eval_data]
predictions = []
for ex in eval_data:
    pred = generate_translation(ex["instruction"])
    predictions.append(pred)

bleu_score = bleu_metric.compute(predictions=predictions, references=[[ref] for ref in references])
print(f"BLEU на валидационном наборе: {bleu_score['score']:.2f}")

for i, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"\nПример {i+1}:")
    print(f"  Инструкция: {eval_data[i]['instruction']}")
    print(f"  Ожидалось : {ref}")
    print(f"  Получено  : {pred}")

print("=" * 70)

# ================================================================
# 23. ТЕСТ НА TRAIN EXAMPLE
# ================================================================

print()
print("=" * 70)
print("TEST: TRAIN EXAMPLE (QLoRA)")
print("=" * 70)

test_instruction = "Переведи на английский: Привет, как дела?"
answer = generate_translation(test_instruction)

print("INPUT :")
print(test_instruction)
print()
print("OUTPUT:")
print(answer)
print("=" * 70)

# ================================================================
# 24. ТЕСТ НА НОВЫХ ПРИМЕРАХ
# ================================================================

test_examples = [
    "Переведи на английский: Я хочу пить.",
    "Переведи на английский: Где находится библиотека?",
    "Переведи на английский: Я люблю Python.",
    "Переведи на английский: До свидания!",
    "Переведи на английский: Как тебя зовут?",
]

print()
print("=" * 70)
print("TEST: NEW EXAMPLES (QLoRA)")
print("=" * 70)

for instruction in test_examples:
    answer = generate_translation(instruction)
    print()
    print("INPUT :", instruction)
    print("OUTPUT:", answer)

print("=" * 70)

In [ ]:
# ================================================================
# ЧИСТЫЙ LoRA FINE-TUNING С SFTTrainer
#
# TinyLlama/TinyLlama-1.1B-Chat-v1.0
#
# Используется:
# - обычная FP16 модель
# - PEFT LoRA
# - TRL SFTTrainer
# - SFTConfig
# ================================================================

# ================================================================
# 1. УСТАНОВКА
# ================================================================
!pip install -q -U transformers peft accelerate datasets trl evaluate sacrebleu matplotlib "numpy<2.1"

# ================================================================
# 2. ИМПОРТЫ
# ================================================================
import os
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    EarlyStoppingCallback,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import evaluate

# ================================================================
# 3. ФИКСАЦИЯ SEED
# ================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ================================================================
# 4. ПРОВЕРКА ОКРУЖЕНИЯ
# ================================================================
print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)
print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("VRAM:", round(vram_gb, 2), "GB")
    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        USE_BF16 = True
        USE_FP16 = False
        print("Dtype: torch.bfloat16")
    else:
        DTYPE = torch.float16
        USE_BF16 = False
        USE_FP16 = True
        print("Dtype: torch.float16")
else:
    DTYPE = torch.float32
    USE_BF16 = False
    USE_FP16 = False
    print("Dtype: torch.float32")
print("=" * 70)

# ================================================================
# 5. КОНФИГУРАЦИЯ
# ================================================================
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./tinyllama_1.1b_lora_translation"
MAX_LENGTH = 256

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# TRAINING
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

# EARLY STOPPING
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# ================================================================
# 6. DATASET
# ================================================================
train_data = [
    {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
    {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
    {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
    {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
    {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
    {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
    {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
    {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
]
eval_data = [
    {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
    {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print("\n" + "=" * 70)
print("DATASET")
print("=" * 70)
print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))
print("\nПример:")
print(train_dataset[0])
print("=" * 70)

# ================================================================
# 7. TOKENIZER
# ================================================================
print("\n" + "=" * 70)
print("ЗАГРУЗКА TOKENIZER")
print("=" * 70)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Vocab size:", tokenizer.vocab_size)
print("PAD token :", tokenizer.pad_token)
print("EOS token :", tokenizer.eos_token)
print("\nChat template:")
print(tokenizer.chat_template)
print("=" * 70)

# ================================================================
# 8. ФОРМИРОВАНИЕ CONVERSATIONAL DATASET
# ================================================================
def convert_to_prompt_completion(example):
    return {
        "prompt": [{"role": "user", "content": example["instruction"]}],
        "completion": [{"role": "assistant", "content": example["output"]}],
    }

train_sft_dataset = train_dataset.map(
    convert_to_prompt_completion,
    remove_columns=train_dataset.column_names,
)
eval_sft_dataset = eval_dataset.map(
    convert_to_prompt_completion,
    remove_columns=eval_dataset.column_names,
)

print("\n" + "=" * 70)
print("SFT DATASET")
print("=" * 70)
print("Train:", len(train_sft_dataset))
print("Eval :", len(eval_sft_dataset))
print("\nПример:")
print(train_sft_dataset[0])
print("=" * 70)

# ================================================================
# 9. ПРОВЕРКА CHAT TEMPLATE
# ================================================================
print("\n" + "=" * 70)
print("ПРОВЕРКА CHAT TEMPLATE")
print("=" * 70)
test_messages = [
    {"role": "user", "content": "Переведи на английский: Привет!"},
    {"role": "assistant", "content": "Hello!"},
]
test_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=False,
)
print(test_text)
print("=" * 70)

# ================================================================
# 10. ЗАГРУЗКА ОБЫЧНОЙ FP16 МОДЕЛИ
# ================================================================
print("\n" + "=" * 70)
print("ЗАГРУЗКА ОБЫЧНОЙ FP16 МОДЕЛИ")
print("=" * 70)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    trust_remote_code=True,
)
model.config.use_cache = False

# ================================================================
# 11. LoRA CONFIG
# ================================================================
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

print("\n" + "=" * 70)
print("LoRA CONFIG")
print("=" * 70)
print("r:", LORA_R)
print("alpha:", LORA_ALPHA)
print("dropout:", LORA_DROPOUT)
print("target_modules:", TARGET_MODULES)
print("=" * 70)

# ================================================================
# 12. SFT CONFIG
# ================================================================
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    optim="adamw_torch",
    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=1,
    logging_first_step=True,
    dataloader_num_workers=0,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
)

# ================================================================
# 13. SFTTrainer
# ================================================================
print("\n" + "=" * 70)
print("СОЗДАНИЕ SFTTrainer")
print("=" * 70)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft_dataset,
    eval_dataset=eval_sft_dataset,
    processing_class=tokenizer,          # современный параметр вместо tokenizer
    peft_config=lora_config,             # SFTTrainer сам применит LoRA
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

# ================================================================
# 14. ПРОВЕРКА TRAINABLE PARAMETERS
# ================================================================
print("\n" + "=" * 70)
print("LoRA TRAINABLE PARAMETERS")
print("=" * 70)
trainer.model.print_trainable_parameters()
print("=" * 70)

# ================================================================
# 15. ПРОВЕРКА TRAINABLE TENSORS
# ================================================================
trainable_tensors = []
for name, parameter in trainer.model.named_parameters():
    if parameter.requires_grad:
        trainable_tensors.append((name, parameter.numel()))
total_trainable = sum(count for _, count in trainable_tensors)
total_parameters = sum(parameter.numel() for parameter in trainer.model.parameters())
trainable_percent = 100.0 * total_trainable / total_parameters

print("\n" + "=" * 70)
print("TRAINABLE PARAMETER CHECK")
print("=" * 70)
print("Trainable parameters:", f"{total_trainable:,}")
print("All parameters:", f"{total_parameters:,}")
print("Trainable %:", f"{trainable_percent:.4f}%")
print("Trainable tensors:", len(trainable_tensors))
print("=" * 70)

# ================================================================
# 16. TRAINING
# ================================================================
print("\n" + "=" * 70)
print("НАЧАЛО LoRA SFT TRAINING")
print("=" * 70)
train_result = trainer.train()
print("\n" + "=" * 70)
print("TRAINING FINISHED")
print("=" * 70)
print(train_result)

# ================================================================
# 17. СОХРАНЕНИЕ LoRA ADAPTER
# ================================================================
print("\n" + "=" * 70)
print("СОХРАНЕНИЕ LoRA ADAPTER")
print("=" * 70)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("\nLoRA adapter сохранён:", OUTPUT_DIR)
print("=" * 70)

# ================================================================
# 18. ПРОВЕРКА ФАЙЛОВ
# ================================================================
print("\n" + "=" * 70)
print("СОДЕРЖИМОЕ OUTPUT DIRECTORY")
print("=" * 70)
if os.path.exists(OUTPUT_DIR):
    for filename in sorted(os.listdir(OUTPUT_DIR)):
        path = os.path.join(OUTPUT_DIR, filename)
        if os.path.isfile(path):
            size_mb = os.path.getsize(path) / 1024**2
            print(f"{filename:40s}{size_mb:10.2f} MB")
print("=" * 70)

# ================================================================
# 19. ГРАФИК TRAINING
# ================================================================
print("\n" + "=" * 70)
print("ГРАФИК ОБУЧЕНИЯ")
print("=" * 70)
log_history = trainer.state.log_history
train_steps, train_losses = [], []
eval_epochs, eval_losses = [], []
for log in log_history:
    if "loss" in log and "step" in log:
        train_steps.append(log["step"])
        train_losses.append(log["loss"])
    if "eval_loss" in log and "epoch" in log:
        eval_epochs.append(log["epoch"])
        eval_losses.append(log["eval_loss"])

plt.figure(figsize=(10, 6))
if train_losses:
    plt.plot(train_steps, train_losses, marker="o", label="Train Loss")
if eval_losses:
    plt.plot(eval_epochs, eval_losses, marker="s", label="Eval Loss")
plt.xlabel("Step / Epoch")
plt.ylabel("Loss")
plt.title("LoRA SFT Training - TinyLlama 1.1B")
plt.grid(True)
plt.legend()
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "training_plot.png")
plt.savefig(plot_path, dpi=150)
plt.show()
print("График сохранён:", plot_path)
print("=" * 70)

# ================================================================
# 20. BLEU
# ================================================================
print("\n" + "=" * 70)
print("ЗАГРУЗКА SACREBLEU")
print("=" * 70)
bleu_metric = evaluate.load("sacrebleu")
print("SacreBLEU загружен.")
print("=" * 70)

# ================================================================
# 21. GENERATION FUNCTION
# ================================================================
def generate_translation(instruction, max_new_tokens=64, temperature=0.2):
    trainer.model.eval()
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt")
    device = next(trainer.model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return answer.strip()

# ================================================================
# 22. ФИНАЛЬНАЯ BLEU ОЦЕНКА
# ================================================================
print("\n" + "=" * 70)
print("ФИНАЛЬНАЯ ОЦЕНКА BLEU")
print("=" * 70)
references = [example["output"] for example in eval_data]
predictions = []
for example in eval_data:
    pred = generate_translation(example["instruction"])
    predictions.append(pred)
bleu_result = bleu_metric.compute(
    predictions=predictions,
    references=[[ref] for ref in references],
)
bleu_score = bleu_result["score"]
print(f"\nBLEU на validation: {bleu_score:.2f}\n")
for i, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"Пример {i + 1}:")
    print("  Instruction:", eval_data[i]["instruction"])
    print("  Reference  :", ref)
    print("  Prediction  :", pred)
    print()
print("=" * 70)

# ================================================================
# 23. TEST TRAIN EXAMPLE
# ================================================================
print("\n" + "=" * 70)
print("TEST: TRAIN EXAMPLE")
print("=" * 70)
test_instruction = "Переведи на английский: Привет, как дела?"
answer = generate_translation(test_instruction)
print("\nINPUT:")
print(test_instruction)
print("\nOUTPUT:")
print(answer)
print("=" * 70)

# ================================================================
# 24. TEST NEW EXAMPLES
# ================================================================
test_examples = [
    "Переведи на английский: Я хочу пить.",
    "Переведи на английский: Где находится библиотека?",
    "Переведи на английский: Я люблю Python.",
    "Переведи на английский: До свидания!",
    "Переведи на английский: Как тебя зовут?",
]
print("\n" + "=" * 70)
print("TEST: NEW EXAMPLES")
print("=" * 70)
for instruction in test_examples:
    answer = generate_translation(instruction)
    print("\nINPUT :", instruction)
    print("OUTPUT:", answer)
print("=" * 70)

# ================================================================
# 25. ФИНАЛЬНАЯ ИНФОРМАЦИЯ
# ================================================================
print("\n" + "=" * 70)
print("ФИНАЛЬНЫЙ РЕЗУЛЬТАТ")
print("=" * 70)
print("\nМетод:\n  LoRA + SFTTrainer")
print("\nМодель:\n", MODEL_NAME)
print("\nQuantization:\n  НЕТ")
print("\nQLoRA:\n  НЕТ")
print("\n4-bit:\n  НЕТ")
print("\nBitsAndBytes:\n  НЕТ")
print("\nPrecision:")
if USE_FP16:
    print("  FP16")
elif USE_BF16:
    print("  BF16")
else:
    print("  FP32")
print("\nLoRA rank:", LORA_R)
print("LoRA alpha:", LORA_ALPHA)
print("LoRA dropout:", LORA_DROPOUT)
print("\nTrain examples:", len(train_data))
print("Eval examples :", len(eval_data))
print("\nOutput:", OUTPUT_DIR)
print("=" * 70)

In [ ]:
# ================================================================
# ПОЛНЫЙ СКВОЗНОЙ ЭКСПЕРИМЕНТ С LoRA / QLoRA
#
# Включает:
# - Выбор метода (LoRA / QLoRA)
# - Профилирование памяти
# - Логирование экспериментов
# - Сравнение с Baseline
# - Визуализация
# ================================================================

# ================================================================
# 1. УСТАНОВКА И ИМПОРТЫ
# ================================================================
!pip install -q -U transformers peft accelerate bitsandbytes datasets trl evaluate sacrebleu matplotlib pandas wandb "numpy<2.1"

import os
import gc
import time
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

import torch
import torch.cuda as cuda
import torch.profiler

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    EarlyStoppingCallback,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
import evaluate

# ================================================================
# 2. КОНФИГУРАЦИЯ
# ================================================================
@dataclass
class ExperimentConfig:
    """Конфигурация эксперимента."""
    # Модель
    model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    method: str = "lora"  # "lora" или "qlora"

    # LoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: List[str] = field(default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj"])

    # Training
    num_epochs: int = 20
    learning_rate: float = 2e-4
    train_batch_size: int = 2
    eval_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    warmup_ratio: float = 0.05
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    max_length: int = 256
    packing: bool = False

    # Early stopping
    early_stopping_patience: int = 3
    early_stopping_threshold: float = 0.001

    # System
    seed: int = 42
    output_dir: str = "./experiment_output"

    # Profile
    profile_steps: int = 5
    log_interval: int = 10

# ================================================================
# 3. ДАТАСЕТ
# ================================================================
def create_dataset():
    """Создаёт небольшой датасет для демонстрации."""
    train_data = [
        {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
        {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
        {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
        {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
        {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
        {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
        {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
        {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
    ]
    eval_data = [
        {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
        {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
    ]
    return Dataset.from_list(train_data), Dataset.from_list(eval_data)

# ================================================================
# 4. BASELINE (Zero-shot / Few-shot)
# ================================================================
def evaluate_baseline(model, tokenizer, eval_data, method="zero_shot"):
    """
    Оценивает Baseline (Zero-shot или Few-shot) без обучения.
    """
    print("\n" + "=" * 70)
    print(f"BASELINE: {method.upper()}")
    print("=" * 70)

    if method == "zero_shot":
        prompt_template = "Переведи на английский: {instruction}"
    else:
        # Few-shot с 2 примерами
        prompt_template = (
            "Пример 1: Переведи на английский: Привет, как дела? -> Hello, how are you?\n"
            "Пример 2: Переведи на английский: Сегодня отличная погода. -> The weather is great today.\n"
            "Теперь переведи: {instruction}"
        )

    predictions = []
    references = []

    for ex in eval_data:
        prompt = prompt_template.format(instruction=ex["instruction"])
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        generated = outputs[0, inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
        predictions.append(answer)
        references.append(ex["output"])

    # BLEU
    bleu_metric = evaluate.load("sacrebleu")
    bleu_score = bleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references]
    )["score"]

    print(f"BLEU: {bleu_score:.2f}")
    for i, (pred, ref) in enumerate(zip(predictions, references)):
        print(f"Пример {i+1}:")
        print(f"  Вопрос: {eval_data[i]['instruction']}")
        print(f"  Ожидалось: {ref}")
        print(f"  Получено: {pred}")
    print("=" * 70)

    return {"bleu": bleu_score, "predictions": predictions}

# ================================================================
# 5. ЗАГРУЗКА МОДЕЛИ (LoRA / QLoRA)
# ================================================================
def load_model_and_tokenizer(config: ExperimentConfig):
    """
    Загружает модель и токенизатор с учётом выбранного метода.
    """
    print("\n" + "=" * 70)
    print(f"ЗАГРУЗКА МОДЕЛИ: {config.method.upper()}")
    print("=" * 70)

    # Определяем dtype
    if torch.cuda.is_available():
        if torch.cuda.is_bf16_supported():
            dtype = torch.bfloat16
            use_bf16 = True
            use_fp16 = False
        else:
            dtype = torch.float16
            use_bf16 = False
            use_fp16 = True
    else:
        dtype = torch.float32
        use_bf16 = False
        use_fp16 = False

    # Токенизатор
    tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Загрузка модели
    if config.method == "qlora":
        # QLoRA с 4-битным квантованием
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=dtype,
        )
        model = AutoModelForCausalLM.from_pretrained(
            config.model_name,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=dtype,
            trust_remote_code=True,
        )
    else:
        # LoRA (обычная точность)
        model = AutoModelForCausalLM.from_pretrained(
            config.model_name,
            torch_dtype=dtype,
            device_map="auto",
            trust_remote_code=True,
        )

    model.config.use_cache = False

    print(f"Метод: {config.method.upper()}")
    print(f"Dtype: {dtype}")
    print(f"Параметров: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
    print("=" * 70)

    return model, tokenizer, use_fp16, use_bf16

# ================================================================
# 6. ПРОФИЛИРОВАНИЕ
# ================================================================
def profile_memory_and_speed(model, tokenizer, dataset, config: ExperimentConfig):
    """
    Профилирует память и скорость обучения.
    """
    print("\n" + "=" * 70)
    print("ПРОФИЛИРОВАНИЕ")
    print("=" * 70)

    # Память до обучения
    if torch.cuda.is_available():
        allocated_before = torch.cuda.memory_allocated() / 1024**3
        print(f"VRAM до загрузки: {allocated_before:.2f} GB")

    # Замер скорости инференса на одном примере
    def measure_inference_speed(n_runs=10):
        times = []
        example = dataset[0]
        prompt = example["instruction"]
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        for _ in range(n_runs):
            torch.cuda.synchronize()
            start = time.time()
            with torch.no_grad():
                model.generate(**inputs, max_new_tokens=32, pad_token_id=tokenizer.pad_token_id)
            torch.cuda.synchronize()
            times.append(time.time() - start)
        return np.mean(times)

    avg_time = measure_inference_speed(5)
    print(f"Среднее время инференса (32 токена): {avg_time*1000:.2f} мс")

    # Профилирование с torch.profiler (если включено)
    if config.profile_steps > 0:
        print(f"Запуск профилирования на {config.profile_steps} шагах...")
        # Здесь можно добавить код профилирования (см. раздел 6.3.2)

    print("=" * 70)
    return {"inference_time_ms": avg_time * 1000}

# ================================================================
# 7. ОБУЧЕНИЕ
# ================================================================
def run_training(config: ExperimentConfig):
    """
    Запускает полный цикл обучения с логированием.
    """
    print("\n" + "=" * 70)
    print("ЗАПУСК ОБУЧЕНИЯ")
    print("=" * 70)

    # 1. Загрузка данных
    train_dataset, eval_dataset = create_dataset()

    # 2. Загрузка модели
    model, tokenizer, use_fp16, use_bf16 = load_model_and_tokenizer(config)

    # 3. Профилирование
    profile_results = profile_memory_and_speed(model, tokenizer, train_dataset, config)

    # 4. Baseline
    print("\n" + "=" * 70)
    print("BASELINE EVALUATION")
    print("=" * 70)
    baseline_results = evaluate_baseline(model, tokenizer, eval_dataset, method="zero_shot")

    # 5. LoRA Config
    lora_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=config.target_modules,
    )

    # 6. Подготовка датасета для SFTTrainer
    def format_conversation(example):
        return {
            "messages": [
                {"role": "user", "content": example["instruction"]},
                {"role": "assistant", "content": example["output"]},
            ]
        }

    train_sft = train_dataset.map(format_conversation, remove_columns=train_dataset.column_names)
    eval_sft = eval_dataset.map(format_conversation, remove_columns=eval_dataset.column_names)

    # 7. SFTConfig
    training_args = SFTConfig(
        output_dir=config.output_dir,
        num_train_epochs=config.num_epochs,
        per_device_train_batch_size=config.train_batch_size,
        per_device_eval_batch_size=config.eval_batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        weight_decay=config.weight_decay,
        max_grad_norm=config.max_grad_norm,
        lr_scheduler_type="cosine",
        warmup_ratio=config.warmup_ratio,
        optim="adamw_torch",
        fp16=use_fp16,
        bf16=use_bf16,
        gradient_checkpointing=True,
        max_length=config.max_length,
        completion_only_loss=True,
        packing=config.packing,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=config.log_interval,
        logging_first_step=True,
        report_to="none",
        remove_unused_columns=False,
        seed=config.seed,
    )

    # 8. SFTTrainer
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_sft,
        eval_dataset=eval_sft,
        processing_class=tokenizer,
        peft_config=lora_config,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=config.early_stopping_patience,
                early_stopping_threshold=config.early_stopping_threshold,
            )
        ],
    )

    # 9. Проверка параметров
    print("\n" + "=" * 70)
    print("TRAINABLE PARAMETERS")
    print("=" * 70)
    trainer.model.print_trainable_parameters()

    # 10. Обучение
    print("\n" + "=" * 70)
    print("НАЧАЛО ОБУЧЕНИЯ")
    print("=" * 70)
    start_time = time.time()
    train_result = trainer.train()
    train_time = time.time() - start_time
    print(f"Обучение завершено за {train_time/60:.2f} минут")

    # 11. Сохранение модели
    trainer.save_model(config.output_dir)
    tokenizer.save_pretrained(config.output_dir)

    # 12. Оценка после обучения
    print("\n" + "=" * 70)
    print("ОЦЕНКА ПОСЛЕ ОБУЧЕНИЯ")
    print("=" * 70)

    # Загружаем лучший чекпоинт (уже загружен благодаря load_best_model_at_end)
    # Оценка на валидации
    bleu_metric = evaluate.load("sacrebleu")
    predictions = []
    references = [ex["output"] for ex in eval_data]

    for ex in eval_data:
        pred = generate_translation(trainer.model, tokenizer, ex["instruction"])
        predictions.append(pred)

    bleu_score = bleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references]
    )["score"]
    print(f"BLEU после обучения: {bleu_score:.2f}")

    # 13. Сбор метрик
    log_history = trainer.state.log_history
    train_losses = [log["loss"] for log in log_history if "loss" in log]
    eval_losses = [(log["epoch"], log["eval_loss"]) for log in log_history if "eval_loss" in log]

    results = {
        "baseline_bleu": baseline_results["bleu"],
        "final_bleu": bleu_score,
        "train_loss": train_losses,
        "eval_loss": eval_losses,
        "train_time_hours": train_time / 3600,
        "profile": profile_results,
        "best_model_path": config.output_dir,
    }

    # 14. Визуализация
    plot_results(train_losses, eval_losses, config.output_dir)

    # 15. Сохранение результатов в таблицу
    save_experiment_results(config, results)

    print("\n" + "=" * 70)
    print("ЭКСПЕРИМЕНТ ЗАВЕРШЁН")
    print("=" * 70)
    print(f"Baseline BLEU: {baseline_results['bleu']:.2f}")
    print(f"LoRA BLEU:     {bleu_score:.2f}")
    print(f"Улучшение:     {bleu_score - baseline_results['bleu']:.2f} пунктов")
    print("=" * 70)

    return results

# ================================================================
# 8. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ================================================================
def generate_translation(model, tokenizer, instruction, max_new_tokens=64):
    """Генерирует перевод с помощью модели."""
    model.eval()
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def plot_results(train_losses, eval_losses, output_dir):
    """Визуализирует графики обучения."""
    plt.figure(figsize=(10, 6))
    if train_losses:
        plt.plot(range(1, len(train_losses)+1), train_losses, marker='o', label='Train Loss')
    if eval_losses:
        eval_epochs, eval_vals = zip(*eval_losses)
        plt.plot(eval_epochs, eval_vals, marker='s', label='Eval Loss')
    plt.xlabel('Step / Epoch')
    plt.ylabel('Loss')
    plt.title('Training Curves')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'training_curves.png'), dpi=150)
    plt.show()

def save_experiment_results(config: ExperimentConfig, results: Dict):
    """Сохраняет результаты в CSV таблицу."""
    # Создаём или загружаем существующую таблицу
    csv_path = "experiments_log.csv"
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
    else:
        df = pd.DataFrame(columns=[
            "id", "date", "model", "method", "r", "lora_alpha", "lr",
            "num_epochs", "batch_size", "baseline_bleu", "final_bleu",
            "train_time_hours", "status", "notes"
        ])

    new_id = len(df) + 1
    new_row = {
        "id": new_id,
        "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "model": config.model_name.split('/')[-1],
        "method": config.method,
        "r": config.lora_r,
        "lora_alpha": config.lora_alpha,
        "lr": config.learning_rate,
        "num_epochs": config.num_epochs,
        "batch_size": config.train_batch_size * config.gradient_accumulation_steps,
        "baseline_bleu": results["baseline_bleu"],
        "final_bleu": results["final_bleu"],
        "train_time_hours": results["train_time_hours"],
        "status": "success",
        "notes": "",
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(csv_path, index=False)
    print(f"✅ Результаты сохранены в {csv_path}")

# ================================================================
# 9. ЗАПУСК ЭКСПЕРИМЕНТА
# ================================================================
if __name__ == "__main__":
    # Создаём конфигурацию
    config = ExperimentConfig(
        model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        method="lora",  # или "qlora"
        lora_r=16,
        lora_alpha=32,
        learning_rate=2e-4,
        num_epochs=3,
        output_dir="./lora_experiment",
    )

    # Запускаем эксперимент
    results = run_training(config)

    # Вывод итогов
    print("\n" + "=" * 70)
    print("ИТОГОВЫЙ РЕЗУЛЬТАТ")
    print("=" * 70)
    print(f"Метод: {config.method.upper()}")
    print(f"Ранг: {config.lora_r}")
    print(f"Baseline BLEU: {results['baseline_bleu']:.2f}")
    print(f"LoRA BLEU:     {results['final_bleu']:.2f}")
    print(f"Улучшение:     {results['final_bleu'] - results['baseline_bleu']:.2f}")
    print(f"Время обучения: {results['train_time_hours']:.2f} ч")
    print("=" * 70)

# Full FTP

In [ ]:
# ================================================================
# ПОЛНАЯ ТОНКАЯ НАСТРОЙКА (FULL FT) НА TINYLLAMA 1.1B
#
# - TinyLlama/TinyLlama-1.1B-Chat-v1.0
# - Перевод с русского на английский
# - Используется SFTTrainer (без LoRA)
# - Оптимизации: gradient checkpointing, FP16, gradient accumulation
# ================================================================

# ================================================================
# 1. УСТАНОВКА
# ================================================================
!pip install -q -U transformers accelerate datasets trl evaluate sacrebleu matplotlib "numpy<2.1"

# ================================================================
# 2. ИМПОРТЫ
# ================================================================
import os
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    EarlyStoppingCallback,
    TrainingArguments,   # <-- Для Full FT используем TrainingArguments
    Trainer,             # <-- Стандартный Trainer (не SFTTrainer, чтобы показать полный контроль)
)
import evaluate

# ================================================================
# 3. ФИКСАЦИЯ SEED
# ================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ================================================================
# 4. ПРОВЕРКА ОКРУЖЕНИЯ
# ================================================================
print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)
print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("VRAM:", round(vram_gb, 2), "GB")
    if torch.cuda.is_bf16_supported():
        DTYPE = torch.bfloat16
        USE_BF16 = True
        USE_FP16 = False
    else:
        DTYPE = torch.float16
        USE_BF16 = False
        USE_FP16 = True
else:
    DTYPE = torch.float32
    USE_BF16 = False
    USE_FP16 = False
print("Dtype:", DTYPE)
print("=" * 70)

# ================================================================
# 5. КОНФИГУРАЦИЯ
# ================================================================
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./tinyllama_fullft_translation"
MAX_LENGTH = 256

# TRAINING
NUM_EPOCHS = 20
LEARNING_RATE = 5e-6               # Full FT требует НИЗКИЙ LR (в 10-100 раз ниже LoRA)
TRAIN_BATCH_SIZE = 1               # Из-за ограниченной памяти
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8    # Эффективный batch size = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

# EARLY STOPPING
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001

# ================================================================
# 6. ДАТАСЕТ
# ================================================================
train_data = [
    {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
    {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
    {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
    {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
    {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
    {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
    {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
    {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
]
eval_data = [
    {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
    {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print("\n" + "=" * 70)
print("DATASET")
print("=" * 70)
print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))
print("\nПример:")
print(train_dataset[0])
print("=" * 70)

# ================================================================
# 7. ТОКЕНИЗАТОР
# ================================================================
print("\n" + "=" * 70)
print("ЗАГРУЗКА ТОКЕНИЗАТОРА")
print("=" * 70)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Vocab size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)
print("=" * 70)

# ================================================================
# 8. ТОКЕНИЗАЦИЯ ДЛЯ FULL FT
# ================================================================
# Для Full FT мы токенизируем данные вручную (без SFTTrainer, чтобы показать полный контроль)

def tokenize_example(example):
    # Формируем промпт с инструкцией
    prompt = f"Инструкция: {example['instruction']}\nОтвет:"
    # Полный текст включает ответ
    full_text = prompt + " " + example['output']

    # Токенизируем весь текст
    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    # Токенизируем только промпт (чтобы вычислить длину промпта для маскировки)
    prompt_tokens = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    prompt_length = len(prompt_tokens["input_ids"])

    # Создаём labels: -100 для токенов промпта, чтобы loss считался только на ответе
    labels = input_ids.copy()
    for i in range(min(prompt_length, len(labels))):
        labels[i] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# Применяем токенизацию
tokenized_train = train_dataset.map(tokenize_example, remove_columns=train_dataset.column_names)
tokenized_eval = eval_dataset.map(tokenize_example, remove_columns=eval_dataset.column_names)

print("\n" + "=" * 70)
print("ТОКЕНИЗАЦИЯ ЗАВЕРШЕНА")
print("=" * 70)
print("Train examples:", len(tokenized_train))
print("Eval examples :", len(tokenized_eval))
print("=" * 70)

# ================================================================
# 9. ЗАГРУЗКА МОДЕЛИ
# ================================================================
print("\n" + "=" * 70)
print("ЗАГРУЗКА МОДЕЛИ (FP16/BF16)")
print("=" * 70)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)

# Включаем gradient checkpointing для экономии памяти
model.gradient_checkpointing_enable()
model.config.use_cache = False

print("Модель загружена.")
print("Параметров:", sum(p.numel() for p in model.parameters()) / 1e9, "B")
print("=" * 70)

# ================================================================
# 10. DATA COLLATOR
# ================================================================
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

# ================================================================
# 11. TRAINING ARGUMENTS
# ================================================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Количество эпох
    num_train_epochs=NUM_EPOCHS,

    # Batch size
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # Learning rate (низкий, чтобы не разрушить предобученные знания)
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    optim="adamw_torch",

    # Точность
    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_checkpointing=True,

    # Логирование и валидация
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
)

# ================================================================
# 12. TRAINER
# ================================================================
print("\n" + "=" * 70)
print("СОЗДАНИЕ TRAINER")
print("=" * 70)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

# ================================================================
# 13. ЗАМЕР ПАМЯТИ ДО ОБУЧЕНИЯ
# ================================================================
if torch.cuda.is_available():
    print("\n" + "=" * 70)
    print("ПАМЯТЬ ДО ОБУЧЕНИЯ")
    print("=" * 70)
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"Выделено: {allocated:.2f} GB")
    print(f"Зарезервировано: {reserved:.2f} GB")
    print("=" * 70)

# ================================================================
# 14. ОБУЧЕНИЕ
# ================================================================
print("\n" + "=" * 70)
print("НАЧАЛО FULL FT TRAINING")
print("=" * 70)
train_result = trainer.train()
print("\n" + "=" * 70)
print("TRAINING FINISHED")
print("=" * 70)
print(train_result)

# ================================================================
# 15. СОХРАНЕНИЕ МОДЕЛИ
# ================================================================
print("\n" + "=" * 70)
print("СОХРАНЕНИЕ МОДЕЛИ")
print("=" * 70)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Модель сохранена в:", OUTPUT_DIR)
print("=" * 70)

# ================================================================
# 16. ГРАФИК ОБУЧЕНИЯ
# ================================================================
print("\n" + "=" * 70)
print("ГРАФИК ОБУЧЕНИЯ")
print("=" * 70)
log_history = trainer.state.log_history
train_steps, train_losses = [], []
eval_epochs, eval_losses = [], []
for log in log_history:
    if "loss" in log and "step" in log:
        train_steps.append(log["step"])
        train_losses.append(log["loss"])
    if "eval_loss" in log and "epoch" in log:
        eval_epochs.append(log["epoch"])
        eval_losses.append(log["eval_loss"])

plt.figure(figsize=(10, 6))
if train_losses:
    plt.plot(train_steps, train_losses, marker="o", label="Train Loss")
if eval_losses:
    plt.plot(eval_epochs, eval_losses, marker="s", label="Eval Loss")
plt.xlabel("Step / Epoch")
plt.ylabel("Loss")
plt.title("Full FT Training - TinyLlama 1.1B")
plt.grid(True)
plt.legend()
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "training_plot.png")
plt.savefig(plot_path, dpi=150)
plt.show()
print("График сохранён:", plot_path)
print("=" * 70)

# ================================================================
# 17. ОЦЕНКА BLEU
# ================================================================
print("\n" + "=" * 70)
print("ОЦЕНКА BLEU")
print("=" * 70)
bleu_metric = evaluate.load("sacrebleu")

def generate_translation(instruction, max_new_tokens=64, temperature=0.0):
    model.eval()
    prompt = f"Инструкция: {instruction}\nОтвет:"
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

references = [ex["output"] for ex in eval_data]
predictions = []
for ex in eval_data:
    pred = generate_translation(ex["instruction"])
    predictions.append(pred)

bleu_score = bleu_metric.compute(predictions=predictions, references=[[ref] for ref in references])["score"]
print(f"BLEU на валидации: {bleu_score:.2f}\n")
for i, (pred, ref) in enumerate(zip(predictions, references)):
    print(f"Пример {i+1}:")
    print("  Instruction:", eval_data[i]["instruction"])
    print("  Reference  :", ref)
    print("  Prediction  :", pred)
print("=" * 70)

# ================================================================
# 18. ТЕСТ НА ОБУЧАЮЩЕМ ПРИМЕРЕ
# ================================================================
print("\n" + "=" * 70)
print("TEST: TRAIN EXAMPLE")
print("=" * 70)
test_instruction = "Переведи на английский: Привет, как дела?"
answer = generate_translation(test_instruction)
print("INPUT :", test_instruction)
print("OUTPUT:", answer)
print("=" * 70)

# ================================================================
# 19. ТЕСТ НА НОВЫХ ПРИМЕРАХ
# ================================================================
test_examples = [
    "Переведи на английский: Я хочу пить.",
    "Переведи на английский: Где находится библиотека?",
    "Переведи на английский: Я люблю Python.",
    "Переведи на английский: До свидания!",
    "Переведи на английский: Как тебя зовут?",
]
print("\n" + "=" * 70)
print("TEST: NEW EXAMPLES")
print("=" * 70)
for instruction in test_examples:
    answer = generate_translation(instruction)
    print("\nINPUT :", instruction)
    print("OUTPUT:", answer)
print("=" * 70)

# ================================================================
# 20. ФИНАЛЬНАЯ ИНФОРМАЦИЯ
# ================================================================
print("\n" + "=" * 70)
print("ФИНАЛЬНЫЙ РЕЗУЛЬТАТ")
print("=" * 70)
print("\nМетод:\n  FULL FINE-TUNING")
print("\nМодель:\n", MODEL_NAME)
print("\nPrecision:")
if USE_FP16:
    print("  FP16")
elif USE_BF16:
    print("  BF16")
else:
    print("  FP32")
print("\nLearning rate:", LEARNING_RATE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("\nTrain examples:", len(train_data))
print("Eval examples :", len(eval_data))
print("\nOutput:", OUTPUT_DIR)
print("=" * 70)

# Dora

In [ ]:
# ================================================================
# СРАВНИТЕЛЬНЫЙ ЭКСПЕРИМЕНТ: LoRA vs QLoRA vs DoRA vs QDoRA
# TinyLlama-1.1B-Chat
#
# Как использовать:
# 1. Запустите все ячейки.
# 2. В конце выберите вариант запуска: один метод или все четыре.
# 3. Для автоматического прогона всех методов раскомментируйте вызов run_all_experiments().
# ================================================================

# ================================================================
# 1. УСТАНОВКА (с обновлением torchao для избежания ошибки)
# ================================================================

!pip install -q -U \
    transformers \
    peft \
    accelerate \
    datasets \
    trl \
    evaluate \
    sacrebleu \
    matplotlib \
    pandas \
    bitsandbytes \
    torchao \
    "numpy<2.1"

# Явно обновляем torchao до совместимой версии
!pip install --upgrade torchao

# ================================================================
# 2. ИМПОРТЫ
# ================================================================

import os
import json
import time
import random
import warnings
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    set_seed,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import evaluate

warnings.filterwarnings("ignore")

# ================================================================
# 3. КОНФИГУРАЦИЯ
# ================================================================

@dataclass
class ExperimentConfig:
    """Конфигурация одного эксперимента."""
    # Model
    model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    # Метод: "lora" или "dora"
    method: str = "lora"          # "lora" | "dora"

    # Использовать 4‑битное квантование? (True → QLoRA/QDoRA)
    use_4bit: bool = False        # True для QLoRA/QDoRA, False для обычных

    # LoRA/DoRA параметры
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: List[str] = field(default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj"])
    bias: str = "none"

    # 4‑bit квантование (актуально, если use_4bit=True)
    bnb_4bit_quant_type: str = "nf4"
    bnb_4bit_use_double_quant: bool = True

    # Training
    num_epochs: int = 3
    learning_rate: float = 2e-4
    train_batch_size: int = 2
    eval_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    warmup_ratio: float = 0.05
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    max_length: int = 256
    packing: bool = False
    optim: str = "adamw_torch"    # "adamw_torch" или "paged_adamw_32bit"

    # Early stopping
    early_stopping_patience: int = 3
    early_stopping_threshold: float = 0.001

    # Generation
    max_new_tokens: int = 64

    # System
    seed: int = 42
    output_dir: str = "./experiment_results"
    logging_steps: int = 1
    profile_inference_runs: int = 5

# ================================================================
# 4. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ================================================================

def get_dtype():
    """Определяет оптимальный dtype для текущего оборудования."""
    if torch.cuda.is_available():
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16, True, False
        else:
            return torch.float16, False, True
    else:
        return torch.float32, False, False

def create_dataset():
    """Создаёт игрушечный датасет для демонстрации."""
    train_data = [
        {"instruction": "Переведи на английский: Привет, как дела?", "output": "Hello, how are you?"},
        {"instruction": "Переведи на английский: Сегодня отличная погода.", "output": "The weather is great today."},
        {"instruction": "Переведи на английский: Я люблю программирование.", "output": "I love programming."},
        {"instruction": "Переведи на английский: Который час?", "output": "What time is it?"},
        {"instruction": "Переведи на английский: Это очень интересно.", "output": "This is very interesting."},
        {"instruction": "Переведи на английский: Доброе утро!", "output": "Good morning!"},
        {"instruction": "Переведи на английский: Спокойной ночи.", "output": "Good night."},
        {"instruction": "Переведи на английский: Как тебя зовут?", "output": "What is your name?"},
    ]
    eval_data = [
        {"instruction": "Переведи на английский: Я хочу пить.", "output": "I am thirsty."},
        {"instruction": "Переведи на английский: Где находится библиотека?", "output": "Where is the library?"},
    ]
    return Dataset.from_list(train_data), Dataset.from_list(eval_data)

def format_conversation(example):
    """Преобразует пример в формат messages для SFTTrainer."""
    return {
        "messages": [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]},
        ]
    }

def generate_text(model, tokenizer, instruction, max_new_tokens=64):
    """Генерирует ответ модели на инструкцию."""
    model.eval()
    messages = [{"role": "user", "content": instruction}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip()

def measure_inference_speed(model, tokenizer, dataset, n_runs=5):
    """Измеряет среднюю скорость инференса."""
    if len(dataset) == 0:
        return {"mean_ms": 0.0, "std_ms": 0.0}

    prompt = dataset[0]["instruction"]
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Warmup
    for _ in range(2):
        model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    times = []
    for _ in range(n_runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.perf_counter() - start)

    return {
        "mean_ms": np.mean(times) * 1000,
        "std_ms": np.std(times) * 1000,
    }

def evaluate_model(model, tokenizer, dataset, max_new_tokens=64):
    """Оценивает модель на датасете (BLEU, Exact Match)."""
    predictions, references = [], []
    for ex in dataset:
        pred = generate_text(model, tokenizer, ex["instruction"], max_new_tokens)
        predictions.append(pred)
        references.append(ex["output"])

    bleu = evaluate.load("sacrebleu").compute(
        predictions=predictions,
        references=[[r] for r in references],
    )["score"]

    em = np.mean([
        p.strip().lower() == r.strip().lower()
        for p, r in zip(predictions, references)
    ]) * 100

    return {"bleu": bleu, "exact_match": em, "predictions": predictions, "references": references}

# ================================================================
# 5. ОСНОВНАЯ ФУНКЦИЯ ЭКСПЕРИМЕНТА
# ================================================================

def run_experiment(config: ExperimentConfig) -> Dict[str, Any]:
    """
    Запускает один эксперимент с заданной конфигурацией.
    Возвращает словарь с результатами.
    """
    # --- Подготовка ---
    set_seed(config.seed)
    dtype, use_bf16, use_fp16 = get_dtype()
    train_dataset, eval_dataset = create_dataset()
    train_sft = train_dataset.map(format_conversation, remove_columns=train_dataset.column_names)
    eval_sft = eval_dataset.map(format_conversation, remove_columns=eval_dataset.column_names)

    method_label = f"{'DoRA' if config.method == 'dora' else 'LoRA'}"
    quant_label = f"{'Q' if config.use_4bit else ''}"
    full_label = f"{method_label}{quant_label}Lora"

    # Создаём папку для результатов
    output_dir = os.path.join(config.output_dir, full_label)
    os.makedirs(output_dir, exist_ok=True)

    # --- Токенизатор ---
    tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # --- Baseline (zero-shot) ---
    print("\n" + "=" * 70)
    print("BASELINE ZERO-SHOT")
    print("=" * 70)
    baseline_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
    )
    baseline_model.config.use_cache = False
    baseline_results = evaluate_model(
        baseline_model, tokenizer, eval_dataset, config.max_new_tokens
    )
    print(f"BLEU: {baseline_results['bleu']:.4f}")
    print(f"Exact Match: {baseline_results['exact_match']:.2f}%")
    print("=" * 70)
    del baseline_model
    torch.cuda.empty_cache()

    # --- Загрузка модели (с квантованием или без) ---
    print("\n" + "=" * 70)
    print(f"ЗАГРУЗКА МОДЕЛИ: {full_label}")
    print("=" * 70)

    if config.use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type=config.bnb_4bit_quant_type,
            bnb_4bit_use_double_quant=config.bnb_4bit_use_double_quant,
            bnb_4bit_compute_dtype=dtype,
        )
        model = AutoModelForCausalLM.from_pretrained(
            config.model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
        print(f"✓ Модель загружена в 4-bit ({config.bnb_4bit_quant_type})")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            config.model_name,
            torch_dtype=dtype,
            device_map="auto",
            trust_remote_code=True,
        )
        print("✓ Модель загружена в полной точности")

    model.config.use_cache = False

    # --- LoRA / DoRA конфиг ---
    peft_config = LoraConfig(
        use_dora=(config.method == "dora"),
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        bias=config.bias,
        target_modules=config.target_modules,
        task_type="CAUSAL_LM",
    )
    print(f"✓ Метод: {full_label} (use_dora={config.method=='dora'}, 4-bit={config.use_4bit})")

    # --- SFTConfig ---
    total_steps = (
        len(train_sft)
        // (config.train_batch_size * config.gradient_accumulation_steps)
    ) * config.num_epochs
    warmup_steps = int(config.warmup_ratio * total_steps)

    training_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=config.num_epochs,
        per_device_train_batch_size=config.train_batch_size,
        per_device_eval_batch_size=config.eval_batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        weight_decay=config.weight_decay,
        max_grad_norm=config.max_grad_norm,
        optim=config.optim,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        fp16=use_fp16,
        bf16=use_bf16,
        gradient_checkpointing=True,
        max_length=config.max_length,
        packing=config.packing,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=config.logging_steps,
        logging_first_step=True,
        report_to="none",
        remove_unused_columns=False,
        seed=config.seed,
        data_seed=config.seed,
        completion_only_loss=True,
    )

    # --- SFTTrainer ---
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_sft,
        eval_dataset=eval_sft,
        processing_class=tokenizer,
        peft_config=peft_config,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=config.early_stopping_patience,
                early_stopping_threshold=config.early_stopping_threshold,
            )
        ],
    )

    # --- Параметры модели ---
    print("\n" + "=" * 70)
    print("ПАРАМЕТРЫ МОДЕЛИ")
    print("=" * 70)
    trainer.model.print_trainable_parameters()
    total_params = sum(p.numel() for p in trainer.model.parameters())
    trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    print(f"Trainable: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.4f}%)")
    print("=" * 70)

    # --- Обучение ---
    print("\n" + "=" * 70)
    print(f"НАЧАЛО ОБУЧЕНИЯ: {full_label}")
    print("=" * 70)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    train_result = trainer.train()
    training_time = time.time() - start_time

    print("\n" + "=" * 70)
    print("ОБУЧЕНИЕ ЗАВЕРШЕНО")
    print("=" * 70)
    print(f"Время: {training_time/60:.2f} мин ({training_time/3600:.4f} ч)")
    print("=" * 70)

    # --- Оценка после обучения ---
    print("\n" + "=" * 70)
    print(f"ОЦЕНКА {full_label}")
    print("=" * 70)

    # Используем лучшую модель (загружена автоматически)
    tuned_results = evaluate_model(
        trainer.model, tokenizer, eval_dataset, config.max_new_tokens
    )
    print(f"BLEU: {tuned_results['bleu']:.4f}")
    print(f"Exact Match: {tuned_results['exact_match']:.2f}%")
    print(f"BLEU прирост: +{tuned_results['bleu'] - baseline_results['bleu']:.4f}")
    print("=" * 70)

    # --- Профилирование инференса ---
    speed = measure_inference_speed(
        trainer.model,
        tokenizer,
        eval_dataset,
        config.profile_inference_runs,
    )
    print("\n" + "=" * 70)
    print("СКОРОСТЬ ИНФЕРЕНСА")
    print("=" * 70)
    print(f"Mean: {speed['mean_ms']:.2f} ms")
    print(f"Std:  {speed['std_ms']:.2f} ms")
    print("=" * 70)

    # --- Сбор метрик ---
    peak_vram = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
    eval_losses = [
        (item.get("epoch", 0), item["eval_loss"])
        for item in trainer.state.log_history
        if "eval_loss" in item
    ]
    best_eval_loss = min([v for _, v in eval_losses]) if eval_losses else None

    results = {
        "method": config.method,
        "use_4bit": config.use_4bit,
        "model": config.model_name,
        "full_label": full_label,
        "rank": config.lora_r,
        "alpha": config.lora_alpha,
        "dropout": config.lora_dropout,
        "target_modules": config.target_modules,
        "train_examples": len(train_dataset),
        "eval_examples": len(eval_dataset),
        "total_parameters": total_params,
        "trainable_parameters": trainable_params,
        "trainable_percent": 100 * trainable_params / total_params,
        "baseline_bleu": baseline_results["bleu"],
        "baseline_exact_match": baseline_results["exact_match"],
        "tuned_bleu": tuned_results["bleu"],
        "tuned_exact_match": tuned_results["exact_match"],
        "bleu_improvement": tuned_results["bleu"] - baseline_results["bleu"],
        "em_improvement": tuned_results["exact_match"] - baseline_results["exact_match"],
        "training_time_seconds": training_time,
        "training_time_hours": training_time / 3600,
        "best_eval_loss": best_eval_loss,
        "peak_vram_gb": peak_vram,
        "inference_mean_ms": speed["mean_ms"],
        "inference_std_ms": speed["std_ms"],
        "train_losses": [item["loss"] for item in trainer.state.log_history if "loss" in item],
        "eval_losses": eval_losses,
        "predictions": tuned_results["predictions"],
        "references": tuned_results["references"],
        "date": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

    # --- Сохранение результатов ---
    with open(os.path.join(output_dir, "results.json"), "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    pd.DataFrame({
        "instruction": [ex["instruction"] for ex in eval_dataset],
        "reference": results["references"],
        "prediction": results["predictions"],
    }).to_csv(os.path.join(output_dir, "predictions.csv"), index=False, encoding="utf-8-sig")

    with open(os.path.join(output_dir, "config.json"), "w", encoding="utf-8") as f:
        config_dict = {k: str(v) for k, v in vars(config).items()}
        json.dump(config_dict, f, ensure_ascii=False, indent=2)

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    # --- Графики ---
    plt.figure(figsize=(10, 6))
    if results["train_losses"]:
        plt.plot(
            range(1, len(results["train_losses"]) + 1),
            results["train_losses"],
            marker="o",
            label="Train Loss",
        )
    if results["eval_losses"]:
        epochs = [x[0] for x in results["eval_losses"] if x[0] is not None]
        values = [x[1] for x in results["eval_losses"] if x[0] is not None]
        plt.plot(epochs, values, marker="s", label="Eval Loss")
    plt.xlabel("Step / Epoch")
    plt.ylabel("Loss")
    plt.title(f"{full_label} Training Curves")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, "training_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # Сравнение BLEU
    plt.figure(figsize=(8, 5))
    plt.bar(
        ["Baseline", full_label],
        [baseline_results["bleu"], tuned_results["bleu"]],
        color=["gray", "#2E86C1"],
    )
    plt.ylabel("BLEU")
    plt.title(f"{full_label} vs Baseline")
    plt.grid(axis="y")
    for i, v in enumerate([baseline_results["bleu"], tuned_results["bleu"]]):
        plt.text(i, v + 0.5, f"{v:.2f}", ha="center", va="bottom")
    plt.savefig(os.path.join(output_dir, "bleu_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # --- Итоговый вывод ---
    print("\n" + "=" * 80)
    print(f"ИТОГОВЫЙ РЕЗУЛЬТАТ: {full_label}")
    print("=" * 80)
    print(f"Модель:         {config.model_name}")
    print(f"Метод:          {full_label}")
    print(f"Trainable %:    {100*trainable_params/total_params:.4f}%")
    print(f"Baseline BLEU:  {baseline_results['bleu']:.4f}")
    print(f"Tuned BLEU:     {tuned_results['bleu']:.4f}  (+{tuned_results['bleu'] - baseline_results['bleu']:.4f})")
    print(f"Exact Match:    {tuned_results['exact_match']:.2f}%  (+{tuned_results['exact_match'] - baseline_results['exact_match']:.2f}%)")
    print(f"Peak VRAM:      {peak_vram:.2f} GB")
    print(f"Training time:  {training_time/3600:.4f} ч ({training_time/60:.2f} мин)")
    print(f"Inference:      {speed['mean_ms']:.2f} ms")
    print(f"Output dir:     {output_dir}")
    print("=" * 80)

    return results


# ================================================================
# 6. АВТОМАТИЧЕСКИЙ ПРОГОН ВСЕХ 4 МЕТОДОВ
# ================================================================

def run_all_experiments(base_config: Optional[ExperimentConfig] = None) -> pd.DataFrame:
    """
    Прогоняет все 4 комбинации: LoRA, QLoRA, DoRA, QDoRA.
    Возвращает DataFrame с результатами.
    """
    if base_config is None:
        base_config = ExperimentConfig()

    all_results = []
    methods = [
        {"method": "lora", "use_4bit": False},
        {"method": "lora", "use_4bit": True},
        {"method": "dora", "use_4bit": False},
        {"method": "dora", "use_4bit": True},
    ]

    for m in methods:
        config = ExperimentConfig(
            model_name=base_config.model_name,
            method=m["method"],
            use_4bit=m["use_4bit"],
            lora_r=base_config.lora_r,
            lora_alpha=base_config.lora_alpha,
            lora_dropout=base_config.lora_dropout,
            target_modules=base_config.target_modules,
            num_epochs=base_config.num_epochs,
            learning_rate=base_config.learning_rate,
            optim=base_config.optim,
        )
        print("\n" + "=" * 80)
        print(f"ЗАПУСК ЭКСПЕРИМЕНТА: {config.method.upper()}{'Q' if config.use_4bit else ''}LoRA")
        print("=" * 80)
        result = run_experiment(config)
        all_results.append(result)
        torch.cuda.empty_cache()

    # Сводим в DataFrame
    df = pd.DataFrame(all_results)
    # Выбираем ключевые колонки
    cols = [
        "full_label",
        "trainable_percent",
        "baseline_bleu",
        "tuned_bleu",
        "bleu_improvement",
        "peak_vram_gb",
        "training_time_hours",
        "inference_mean_ms",
    ]
    return df[cols]


# ================================================================
# 7. ЗАПУСК
# ================================================================

if __name__ == "__main__":
    # Вариант А: запустить один метод (измените config)
    config = ExperimentConfig(
        method="lora",          # или "dora"
        use_4bit=False,         # или True
        optim="adamw_torch",    # или "paged_adamw_32bit"
    )
    results = run_experiment(config)

    # Вариант Б: автоматически прогнать все 4 метода
    # df = run_all_experiments(ExperimentConfig())
    # print("\nСРАВНЕНИЕ ВСЕХ МЕТОДОВ:")
    # print(df.to_string(index=False))